In [1]:
!pip -q install xgboost shap plotly streamlit networkx scipy joblib
import os; os.makedirs('/content/ids_src', exist_ok=True)

In [2]:
%%writefile /content/ids_src/core.py
"""core.py - configuration, logging and the single FeatureSchema used everywhere."""
from __future__ import annotations
import hashlib, json, logging, os
from dataclasses import dataclass, field, asdict
from typing import Dict, List
import numpy as np
import pandas as pd

def get_logger(name: str = "ids") -> logging.Logger:
    lg = logging.getLogger(name)
    if not lg.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s", "%H:%M:%S"))
        lg.addHandler(h); lg.setLevel(logging.INFO)
    return lg

@dataclass
class Config:
    """Central configuration (all tunables live here)."""
    seed: int = 42
    work_dir: str = "/content/ids_artifacts"
    data_dir: str = "/content/cicids2017"
    max_rows_per_class: int = 200_000      # Colab Free RAM guard (time-ordered systematic sampling)
    train_cap_per_class: int = 40_000      # capped undersampling for supervised training (RAM/time)
    min_class_rows: int = 200              # classes below this are excluded from supervised training
    split: tuple = (0.6, 0.2, 0.2)         # chronological blocks per class
    split_gap: float = 0.01                # fraction of rows dropped between blocks (leakage buffer)
    target_fpr_review: float = 0.02
    target_fpr_high: float = 0.002
    signal_quantile: float = 0.99          # anomaly signal = above 99th pct of benign validation scores
    flow_timeout_s: float = 120.0
    activity_timeout_us: float = 5_000_000.0
    temporal_windows_s: tuple = (10.0, 60.0, 300.0)
    temporal_consecutive: int = 3
    auto_response: bool = False
    rule_ttl_s: float = 600.0

# --------------------------------------------------------------------------- schema
# (name, unit, kind) ; kind in {count,bytes,time_us,rate,ratio,flag}. Names are the CIC-IDS2017 names.
_FEATURES = [
 ("Flow Duration","us","time_us"),("Total Fwd Packets","pkts","count"),("Total Backward Packets","pkts","count"),
 ("Total Length of Fwd Packets","bytes","bytes"),("Total Length of Bwd Packets","bytes","bytes"),
 ("Fwd Packet Length Max","bytes","bytes"),("Fwd Packet Length Min","bytes","bytes"),
 ("Fwd Packet Length Mean","bytes","bytes"),("Fwd Packet Length Std","bytes","bytes"),
 ("Bwd Packet Length Max","bytes","bytes"),("Bwd Packet Length Min","bytes","bytes"),
 ("Bwd Packet Length Mean","bytes","bytes"),("Bwd Packet Length Std","bytes","bytes"),
 ("Flow Bytes/s","bytes/s","rate"),("Flow Packets/s","pkts/s","rate"),
 ("Flow IAT Mean","us","time_us"),("Flow IAT Std","us","time_us"),("Flow IAT Max","us","time_us"),("Flow IAT Min","us","time_us"),
 ("Fwd IAT Total","us","time_us"),("Fwd IAT Mean","us","time_us"),("Fwd IAT Std","us","time_us"),("Fwd IAT Max","us","time_us"),("Fwd IAT Min","us","time_us"),
 ("Bwd IAT Total","us","time_us"),("Bwd IAT Mean","us","time_us"),("Bwd IAT Std","us","time_us"),("Bwd IAT Max","us","time_us"),("Bwd IAT Min","us","time_us"),
 ("Fwd PSH Flags","count","flag"),("Fwd Packets/s","pkts/s","rate"),("Bwd Packets/s","pkts/s","rate"),
 ("Min Packet Length","bytes","bytes"),("Max Packet Length","bytes","bytes"),("Packet Length Mean","bytes","bytes"),
 ("Packet Length Std","bytes","bytes"),("Packet Length Variance","bytes^2","bytes"),
 ("FIN Flag Count","count","flag"),("SYN Flag Count","count","flag"),("RST Flag Count","count","flag"),
 ("PSH Flag Count","count","flag"),("ACK Flag Count","count","flag"),("URG Flag Count","count","flag"),
 ("Down/Up Ratio","ratio","ratio"),("Average Packet Size","bytes","bytes"),
 ("Avg Fwd Segment Size","bytes","bytes"),("Avg Bwd Segment Size","bytes","bytes"),
 ("Active Mean","us","time_us"),("Active Std","us","time_us"),("Active Max","us","time_us"),("Active Min","us","time_us"),
 ("Idle Mean","us","time_us"),("Idle Std","us","time_us"),("Idle Max","us","time_us"),("Idle Min","us","time_us"),
]
RATE_COLS = ["Flow Bytes/s","Flow Packets/s","Fwd Packets/s","Bwd Packets/s"]

class FeatureSchema:
    """Single source of truth for model features (Tier A: exactly derivable from simulated packets).

    Excluded on purpose (documented, never zero-filled): Destination Port (shortcut feature), header-length,
    Init_Win_bytes_*, bulk/subflow features, act_data_pkt_fwd, min_seg_size_forward, ECE/CWE flags.
    Zero-duration rule (shared by dataset + extractor): rates are 0 when Flow Duration <= 0.
    Units follow CICFlowMeter: durations/IATs in microseconds, rates per second, lengths = payload bytes.
    """
    names: List[str] = [f[0] for f in _FEATURES]
    units: Dict[str, str] = {f[0]: f[1] for f in _FEATURES}
    kinds: Dict[str, str] = {f[0]: f[2] for f in _FEATURES}
    dtype = np.float64
    EXCLUDED = ["Destination Port","Fwd Header Length","Init_Win_bytes_forward","Init_Win_bytes_backward",
                "act_data_pkt_fwd","min_seg_size_forward","Bwd Header Length","ECE Flag Count","CWE Flag Count"]

    @classmethod
    def version_hash(cls) -> str:
        return hashlib.sha256(json.dumps(cls.names).encode()).hexdigest()[:12]

    @classmethod
    def formulas(cls) -> Dict[str, str]:
        return {"Flow Duration":"last_ts-first_ts (us)","Flow IAT *":"diff of sorted packet timestamps (us)",
                "Fwd/Bwd IAT *":"diff of timestamps within a direction (us)","Flow Bytes/s":"(fwd_len+bwd_len)/(duration/1e6)",
                "Flow Packets/s":"n_packets/(duration/1e6)","*Flag Count":"sum over packets of the TCP flag bit",
                "Packet Length Std":"sample std (ddof=1) over payload lengths","Down/Up Ratio":"floor(bwd_pkts/fwd_pkts)",
                "Active/Idle *":"gaps > activity timeout split active/idle periods"}

    # ---- shared preprocessing step 1 (training AND inference) --------------------------------
    @classmethod
    def validate(cls, df: pd.DataFrame, strict: bool = True) -> pd.DataFrame:
        """Return df restricted to schema columns, in schema order, float64. Raises on missing columns."""
        missing = [c for c in cls.names if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required features: {missing}")
        return df[cls.names].astype(cls.dtype)

    @classmethod
    def apply_zero_duration_rule(cls, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        z = df["Flow Duration"] <= 0
        for c in RATE_COLS:
            df.loc[z | ~np.isfinite(df[c]), c] = 0.0
        return df

    @classmethod
    def clean(cls, df: pd.DataFrame) -> pd.DataFrame:
        """Identical cleaning for CIC rows and simulator rows: schema order, zero-duration rule, finite check."""
        x = cls.apply_zero_duration_rule(cls.validate(df))
        x = x.replace([np.inf, -np.inf], np.nan)
        return x

    @classmethod
    def valid_mask(cls, x: pd.DataFrame) -> pd.Series:
        """Rows that are finite and non-negative (negatives are CICFlowMeter artifacts)."""
        return x.notna().all(axis=1) & (x >= 0).all(axis=1)

class Preprocessor:
    """Shared preprocessing: log1p -> standardize. Fit on TRAIN only; same object at inference."""
    def __init__(self): self.mu = None; self.sd = None; self.schema_hash = FeatureSchema.version_hash()
    @staticmethod
    def _log(x: np.ndarray) -> np.ndarray: return np.log1p(np.clip(x, 0, None))
    def fit(self, X: pd.DataFrame):
        L = self._log(FeatureSchema.validate(X).values)
        self.mu = L.mean(0); self.sd = L.std(0); self.sd[self.sd < 1e-9] = 1.0
        return self
    def transform(self, X) -> np.ndarray:
        v = FeatureSchema.validate(X).values if isinstance(X, pd.DataFrame) else np.asarray(X, float)
        if not np.isfinite(v).all(): raise ValueError("Non-finite values reached preprocessing")
        return np.clip((self._log(v) - self.mu) / self.sd, -10, 10).astype(np.float32)

Overwriting /content/ids_src/core.py


In [3]:
%%writefile /content/ids_src/sim.py
"""sim.py - custom Python network simulator (SIMULATED traffic only; no OS packets are ever created)."""
from __future__ import annotations
import heapq, ipaddress, itertools
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
import numpy as np
import networkx as nx
from core import get_logger, Config

log = get_logger("ids.sim")
FIN, SYN, RST, PSH, ACK, URG = 1, 2, 4, 8, 16, 32
HDR = {"TCP": 40, "UDP": 28, "ICMP": 28}
MSS = 1460

# ------------------------------------------------------------------ data structures
@dataclass
class Packet:
    """One simulated packet event as captured by the monitoring node."""
    ts: float; src_ip: str; dst_ip: str; sport: int; dport: int; proto: str
    size: int; payload: int; flags: int; conn_id: str; direction: str
    src_dev: str; dst_dev: str; label: str = "normal"; meta: Dict[str, Any] = field(default_factory=dict)
    simulated: bool = True

@dataclass
class Interface:
    name: str; ip: str; mac: str; segment: str; prefix: int = 24

class Device:
    """Base network device."""
    dtype = "device"
    def __init__(self, dev_id: str, name: str, segment: str):
        self.id, self.name, self.segment = dev_id, name, segment
        self.interfaces: List[Interface] = []
        self.services: Dict[int, str] = {}
        self.connection_state = "up"          # up | down
        self.security_status = "normal"       # normal | blocked | isolated | compromised
    @property
    def ip(self) -> Optional[str]: return self.interfaces[0].ip if self.interfaces else None
    @property
    def mac(self) -> Optional[str]: return self.interfaces[0].mac if self.interfaces else None
    def add_interface(self, name, ip, mac, segment, prefix=24): self.interfaces.append(Interface(name, ip, mac, segment, prefix))
    def to_dict(self): return dict(id=self.id, name=self.name, type=self.dtype, ip=self.ip, mac=self.mac, segment=self.segment,
                                    status=self.security_status, state=self.connection_state, services=self.services)

class Host(Device): dtype = "host"
class Client(Host): dtype = "client"
class Server(Host): dtype = "server"
class Attacker(Host): dtype = "attacker"
class MonitoringNode(Device):
    dtype = "monitor"
    def __init__(self, *a, **k): super().__init__(*a, **k); self.captured = 0

class Switch(Device):
    """Learning switch: MAC table, unicast when known else flood."""
    dtype = "switch"
    def __init__(self, *a, **k): super().__init__(*a, **k); self.mac_table: Dict[str, str] = {}; self.floods = 0; self.unicasts = 0
    def forward(self, src_mac: str, dst_mac: str, ingress: str) -> str:
        self.mac_table[src_mac] = ingress
        if dst_mac in self.mac_table: self.unicasts += 1; return "unicast"
        self.floods += 1; return "flood"

@dataclass
class ACLRule:
    rule_id: int; action: str; src: str = "any"; dst: str = "any"; proto: str = "any"; dport: Optional[int] = None
    hard: bool = False            # hard rules also drop packets of already-established connections
    expires: Optional[float] = None; comment: str = ""; created: float = 0.0; hits: int = 0
    def matches(self, p: Packet) -> bool:
        def inn(ip, net): return net == "any" or ipaddress.ip_address(ip) in ipaddress.ip_network(net, strict=False)
        return (inn(p.src_ip, self.src) and inn(p.dst_ip, self.dst) and self.proto in ("any", p.proto)
                and (self.dport is None or self.dport == p.dport))

class Router(Device):
    """Router with longest-prefix routing and an ordered ACL (first match wins, default permit)."""
    dtype = "router"
    def __init__(self, *a, **k):
        super().__init__(*a, **k); self.acl: List[ACLRule] = []; self._ids = itertools.count(1); self.forwarded = 0; self.denied = 0
    def route(self, dst_ip: str) -> Optional[Interface]:
        best, bl = None, -1
        for itf in self.interfaces:
            net = ipaddress.ip_network(f"{itf.ip}/{itf.prefix}", strict=False)
            if ipaddress.ip_address(dst_ip) in net and net.prefixlen > bl: best, bl = itf, net.prefixlen
        return best
    def add_rule(self, action, now=0.0, ttl=None, **kw) -> ACLRule:
        r = ACLRule(next(self._ids), action, created=now, expires=(now + ttl) if ttl else None, **kw); self.acl.append(r); return r
    def remove_rule(self, rule_id: int) -> bool:
        n = len(self.acl); self.acl = [r for r in self.acl if r.rule_id != rule_id]; return len(self.acl) < n
    def expire(self, now: float) -> List[ACLRule]:
        gone = [r for r in self.acl if r.expires is not None and r.expires <= now]
        self.acl = [r for r in self.acl if r not in gone]; return gone
    def check(self, p: Packet, established: bool) -> Tuple[str, Optional[ACLRule]]:
        for r in self.acl:
            if established and not r.hard: continue
            if r.matches(p): r.hits += 1; return r.action, r
        return "permit", None

# ------------------------------------------------------------------ simulator
class NetworkSimulator:
    """Topology (NetworkX) + packet forwarding + connection tracking + capture at the monitoring tap."""
    def __init__(self, seed: int = 42, loss: float = 0.0, jitter: float = 0.1):
        self.seed, self.loss, self.jitter = seed, loss, jitter
        self.reset(seed)
    def reset(self, seed: Optional[int] = None):
        if seed is not None: self.seed = seed
        self.rng = np.random.default_rng(self.seed)
        self.G = nx.Graph(); self.dev: Dict[str, Device] = {}; self.by_ip: Dict[str, str] = {}
        self.clock = 0.0; self.pending: List[Tuple[float, int, Packet]] = []; self._seq = itertools.count()
        self.conn_table: Dict[str, bool] = {}; self._cid = itertools.count(1); self.drop_log: List[dict] = []
        self.stats = dict(delivered=0, dropped=0, sessions=0); self._path_cache: Dict[Tuple[str, str], Optional[List[str]]] = {}
        self.build_default_topology()

    # ---- topology ---------------------------------------------------------------------
    def _add(self, d: Device, **pos):
        self.dev[d.id] = d; self.G.add_node(d.id, **pos)
        for i in d.interfaces: self.by_ip[i.ip] = d.id
    def _link(self, a, b, lat): self.G.add_edge(a, b, latency=lat); self._path_cache.clear()
    @staticmethod
    def _mac(n): return "02:00:00:00:%02x:%02x" % (n // 256, n % 256)
    def build_default_topology(self):
        n = itertools.count(1)
        r = Router("R1", "Router-1", "core")
        for nm, ip, seg in [("eth0", "192.168.10.1", "LAN"), ("eth1", "192.168.20.1", "DMZ"), ("eth2", "10.0.0.1", "EXT"), ("eth3", "192.168.30.1", "MGMT")]:
            r.add_interface(nm, ip, self._mac(next(n)), seg)
        self._add(r, x=0, y=0)
        for sid, seg, x, y in [("SW1", "LAN", -3, -1.5), ("SW2", "DMZ", 0, -3), ("SW3", "EXT", 3, -1.5)]:
            s = Switch(sid, sid, seg); s.add_interface("sw", f"0.0.0.{len(self.dev)}", self._mac(next(n)), seg); self._add(s, x=x, y=y)
        m = MonitoringNode("MON", "IDS-Monitor", "MGMT"); m.add_interface("eth0", "192.168.30.2", self._mac(next(n)), "MGMT"); self._add(m, x=0, y=1.6)
        for i in range(1, 7):
            c = Client(f"c{i}", f"Client-{i}", "LAN"); c.add_interface("eth0", f"192.168.10.{10+i}", self._mac(next(n)), "LAN")
            self._add(c, x=-5 + (i - 1) * 0.55, y=-3.4)
        for sid, nm, last, svc, x in [("web", "Web-Server", 10, {80: "http", 443: "https"}, -1.6), ("dns", "DNS-Server", 11, {53: "dns"}, -0.55),
                                       ("ftp", "FTP-Server", 12, {21: "ftp"}, 0.55), ("ssh", "SSH-Server", 13, {22: "ssh"}, 1.6)]:
            s = Server(sid, nm, "DMZ"); s.add_interface("eth0", f"192.168.20.{last}", self._mac(next(n)), "DMZ"); s.services = svc; self._add(s, x=x, y=-4.6)
        for i in range(1, 4):
            a = Attacker(f"a{i}", f"Attacker-{i}", "EXT"); a.add_interface("eth0", f"10.0.0.{100+i}", self._mac(next(n)), "EXT"); self._add(a, x=4.5 + (i - 1) * 0.7, y=-3.4)
        c2 = Server("cnc", "Botnet-C2", "EXT"); c2.add_interface("eth0", "10.0.0.50", self._mac(next(n)), "EXT"); c2.services = {8080: "c2-http"}; self._add(c2, x=6.8, y=-0.5)
        for i in range(1, 7): self._link(f"c{i}", "SW1", 0.0001)
        for s in ("web", "dns", "ftp", "ssh"): self._link(s, "SW2", 0.0001)
        for a in ("a1", "a2", "a3", "cnc"): self._link(a, "SW3", 0.0002)
        self._link("SW1", "R1", 0.0002); self._link("SW2", "R1", 0.0002); self._link("SW3", "R1", 0.008); self._link("MON", "R1", 0.00005)
        R = self.dev["R1"]
        R.add_rule("deny", src="10.0.0.0/24", dst="192.168.10.0/24", comment="baseline: EXT cannot initiate to LAN")
        R.add_rule("deny", src="10.0.0.0/24", dst="192.168.30.0/24", comment="baseline: EXT cannot reach MGMT")
        self.router = R

    # ---- device / path helpers ---------------------------------------------------------
    def _active(self, d: str) -> bool:
        x = self.dev[d]; return x.connection_state == "up" and x.security_status != "isolated"
    def path(self, a: str, b: str) -> Optional[List[str]]:
        k = (a, b)
        if k in self._path_cache: return self._path_cache[k]
        H = self.G.subgraph([n for n in self.G.nodes if self._active(n)])
        try: p = nx.shortest_path(H, a, b, weight="latency")
        except (nx.NetworkXNoPath, nx.NodeNotFound): p = None
        self._path_cache[k] = p; return p
    def isolate(self, dev_id: str, on: bool = True):
        self.dev[dev_id].security_status = "isolated" if on else "normal"; self._path_cache.clear()
    def one_way(self, a: str, b: str) -> Optional[float]:
        p = self.path(a, b)
        return None if p is None else sum(self.G[u][v]["latency"] for u, v in zip(p, p[1:]))
    def segment_of(self, ip): return self.dev[self.by_ip[ip]].segment
    def new_cid(self, proto, a, ap, b, bp): return f"{proto}-{a}:{ap}-{b}:{bp}-{next(self._cid)}"

    # ---- packet delivery ----------------------------------------------------------------
    def deliver(self, t: float, src: str, dst: str, sport: int, dport: int, proto: str, flags: int, payload: int,
                cid: str, label: str = "normal", meta: Optional[dict] = None) -> Tuple[bool, float, str]:
        """Forward one packet hop by hop. Returns (delivered, arrival_time, reason). Captures at the tap after ACL."""
        sd, dd = self.dev[src], self.dev[dst]
        size = payload + HDR[proto]
        def drop(reason, rule=None):
            self.stats["dropped"] += 1
            self.drop_log.append(dict(ts=t, src_ip=sd.ip, dst_ip=dd.ip, dport=dport, proto=proto, reason=reason, rule=getattr(rule, "rule_id", None)))
            return False, t, reason
        if not (self._active(src) and self._active(dst)): return drop("isolated-or-down")
        p = self.path(src, dst)
        if p is None: return drop("no-route")
        if self.loss and self.rng.random() < self.loss: return drop("loss")
        sip, dip = sd.ip, dd.ip
        direction = ("internal" if self.segment_of(sip) == self.segment_of(dip) or {self.segment_of(sip), self.segment_of(dip)} <= {"LAN", "DMZ"}
                     else ("inbound" if self.segment_of(sip) == "EXT" else "outbound"))
        pkt = Packet(0.0, sip, dip, sport, dport, proto, size, payload, flags, cid, direction, src, dst, label, meta or {})
        established = cid in self.conn_table
        now, captured, prev = t, False, p[0]
        for node in p[1:-1]:
            now += self.G[prev][node]["latency"] * max(0.2, 1 + self.rng.normal(0, self.jitter))
            d = self.dev[node]
            if isinstance(d, Switch): d.forward(sd.mac, dd.mac, prev)
            elif isinstance(d, Router):
                if d.route(dip) is None: return drop("no-route-to-host")
                act, rule = d.check(pkt, established)
                if act == "deny": d.denied += 1; return drop("acl-deny", rule)
                d.forwarded += 1
            if not captured: pkt.ts = now; captured = True
            prev = node
        now += self.G[prev][dst]["latency"]
        if not captured: pkt.ts = now
        is_new = (proto == "TCP" and (flags & SYN) and not (flags & ACK)) or (proto != "TCP" and not established)
        if is_new: self.conn_table[cid] = True
        heapq.heappush(self.pending, (pkt.ts, next(self._seq), pkt))
        self.stats["delivered"] += 1; self.dev["MON"].captured += 1
        return True, now, "ok"

    def harvest(self, until: Optional[float] = None) -> List[Packet]:
        """Pop captured packets with ts <= until (all if None), in timestamp order."""
        out = []
        while self.pending and (until is None or self.pending[0][0] <= until): out.append(heapq.heappop(self.pending)[2])
        return out

    # ---- protocol behaviours ------------------------------------------------------------
    def eph(self) -> int: return int(self.rng.integers(32768, 61000))
    def _segs(self, n: int) -> List[int]:
        if n <= 0: return [0]
        out = [MSS] * (n // MSS); r = n % MSS
        if r: out.append(r)
        return out
    def tcp_session(self, t0: float, src: str, dst: str, dport: int, exchanges: List[Tuple[int, int, float]], close: str = "fin",
                    label: str = "normal", sport: Optional[int] = None, handshake: bool = True, meta: Optional[dict] = None,
                    proc: float = 0.00005) -> dict:
        """Full TCP lifecycle: 3-way handshake, request/response exchanges (req_bytes, resp_bytes, think_s), teardown (fin|rst|none).
        Closed port -> RST/ACK. ACL drop -> no reply. Returns summary dict incl. 'established' and 'end' time."""
        sp = sport or self.eph(); dd = self.dev[dst]; cid = self.new_cid("TCP", self.dev[src].ip, sp, dd.ip, dport)
        self.stats["sessions"] += 1
        def S(t, a, b, s, d, fl, pl): return self.deliver(t, a, b, s, d, "TCP", fl, pl, cid, label, meta)
        ok, t, why = S(t0, src, dst, sp, dport, SYN, 0)
        if not ok: return dict(established=False, reason=why, end=t0, cid=cid)
        t += proc
        if dport not in dd.services:                                  # closed port
            ok, t, why = S(t, dst, src, dport, sp, RST | ACK, 0)
            return dict(established=False, reason="closed-port", end=t, cid=cid)
        if not handshake:                                             # half-open scan: SYN/ACK then client RST
            ok, t, _ = S(t, dst, src, dport, sp, SYN | ACK, 0); t += proc
            ok, t, _ = S(t, src, dst, sp, dport, RST, 0)
            return dict(established=False, reason="half-open", end=t, cid=cid)
        ok, t, _ = S(t, dst, src, dport, sp, SYN | ACK, 0); t += proc
        ok, t, _ = S(t, src, dst, sp, dport, ACK, 0); t += proc
        for req, resp, think in exchanges:
            t += think
            segs = self._segs(req)
            for i, s in enumerate(segs):
                ok, t2, _ = S(t, src, dst, sp, dport, (PSH | ACK) if i == len(segs) - 1 else ACK, s); t = t2 + proc * 0.2
            ok, t, _ = S(t + proc, dst, src, dport, sp, ACK, 0); t += proc
            if resp > 0:
                segs = self._segs(resp)
                for i, s in enumerate(segs):
                    ok, t2, _ = S(t, dst, src, dport, sp, (PSH | ACK) if i == len(segs) - 1 else ACK, s); t = t2 + proc * 0.2
                    if i % 2 == 1 or i == len(segs) - 1: ok, t3, _ = S(t2 + proc, src, dst, sp, dport, ACK, 0); t = max(t, t3) + proc * 0.2
        if close == "fin":
            ok, t, _ = S(t + proc, src, dst, sp, dport, FIN | ACK, 0); t += proc
            ok, t, _ = S(t, dst, src, dport, sp, ACK, 0); t += proc
            ok, t, _ = S(t, dst, src, dport, sp, FIN | ACK, 0); t += proc
            ok, t, _ = S(t, src, dst, sp, dport, ACK, 0)
        elif close == "rst":
            ok, t, _ = S(t + proc, src, dst, sp, dport, RST | ACK, 0)
        return dict(established=True, reason="ok", end=t, cid=cid, sport=sp)

    def udp_exchange(self, t0, src, dst, dport, req, resp, label="normal", sport=None, rtt_extra=0.0005, meta=None) -> dict:
        sp = sport or self.eph(); dd = self.dev[dst]; cid = self.new_cid("UDP", self.dev[src].ip, sp, dd.ip, dport)
        ok, t, why = self.deliver(t0, src, dst, sp, dport, "UDP", 0, req, cid, label, meta)
        if not ok: return dict(ok=False, reason=why, end=t0)
        if dport in dd.services and resp > 0:
            ok, t, _ = self.deliver(t + rtt_extra, dst, src, dport, sp, "UDP", 0, resp, cid, label, meta)
        return dict(ok=ok, reason="ok", end=t)
    def icmp_echo(self, t0, src, dst, payload=56, count=1, interval=1.0, label="normal", meta=None) -> dict:
        cid = self.new_cid("ICMP", self.dev[src].ip, 0, self.dev[dst].ip, 0); t = t0; got = 0
        for _ in range(count):
            ok, ta, why = self.deliver(t, src, dst, 0, 0, "ICMP", 0, payload, cid, label, meta)
            if ok:
                ok2, _, _ = self.deliver(ta + 0.00002, dst, src, 0, 0, "ICMP", 0, payload, cid, label, meta); got += int(ok2)
            t += interval
        return dict(ok=got > 0, replies=got, end=t)
    def dns_query(self, t0, client, server="dns", label="normal"):
        return self.udp_exchange(t0, client, server, 53, int(self.rng.integers(28, 70)), int(self.rng.integers(60, 400)), label, meta={"app": "dns"})
    def http_request(self, t0, client, server="web", req=400, resp=5000, label="normal", close="fin", port=80):
        return self.tcp_session(t0, client, server, port, [(req, resp, 0.0)], close, label, meta={"app": "http"})

    def probe(self, src: str, dst: str, proto: str = "TCP", dport: int = 80) -> dict:
        """Mitigation verification: attempt communication now and report whether it works. Packets are captured (label 'probe')."""
        t = self.clock + 1e-3
        if proto == "ICMP": r = self.icmp_echo(t, src, dst, 32, 1, label="probe"); return dict(reachable=r["ok"], reason="ok" if r["ok"] else "no-reply")
        if proto == "UDP": r = self.udp_exchange(t, src, dst, dport, 40, 80, "probe"); return dict(reachable=bool(r["ok"]), reason=r["reason"])
        r = self.tcp_session(t, src, dst, dport, [(100, 100, 0.0)], "fin", "probe"); return dict(reachable=r["established"], reason=r["reason"])

    # ---- visualization ----------------------------------------------------------------
    def topology_figure(self, highlight_recent: int = 0):
        import plotly.graph_objects as go
        col = dict(client="#3b82f6", server="#10b981", attacker="#ef4444", router="#f59e0b", switch="#8b5cf6", monitor="#14b8a6", host="#64748b")
        ex, ey = [], []
        for u, v in self.G.edges:
            if self._active(u) and self._active(v): ex += [self.G.nodes[u]["x"], self.G.nodes[v]["x"], None]; ey += [self.G.nodes[u]["y"], self.G.nodes[v]["y"], None]
        dx, dy = [], []
        for u, v in self.G.edges:
            if not (self._active(u) and self._active(v)): dx += [self.G.nodes[u]["x"], self.G.nodes[v]["x"], None]; dy += [self.G.nodes[u]["y"], self.G.nodes[v]["y"], None]
        fig = go.Figure(go.Scatter(x=ex, y=ey, mode="lines", line=dict(width=1.5, color="#94a3b8"), hoverinfo="none", name="active link"))
        if dx: fig.add_trace(go.Scatter(x=dx, y=dy, mode="lines", line=dict(width=1.5, color="#ef4444", dash="dot"), hoverinfo="none", name="cut link"))
        for dt in col:
            ids = [i for i, d in self.dev.items() if d.dtype == dt]
            if not ids: continue
            fig.add_trace(go.Scatter(x=[self.G.nodes[i]["x"] for i in ids], y=[self.G.nodes[i]["y"] for i in ids], mode="markers+text",
                text=[self.dev[i].name for i in ids], textposition="bottom center",
                hovertext=[f"{self.dev[i].name}<br>{self.dev[i].ip}<br>{self.dev[i].segment}<br>status={self.dev[i].security_status}" for i in ids],
                marker=dict(size=[26 if self.dev[i].dtype == "router" else 18 for i in ids], color=col[dt],
                            line=dict(width=[4 if self.dev[i].security_status != "normal" else 1 for i in ids],
                                      color=["#dc2626" if self.dev[i].security_status != "normal" else "#0f172a" for i in ids])), name=dt))
        fig.update_layout(height=520, margin=dict(l=10, r=10, t=30, b=10), xaxis=dict(visible=False), yaxis=dict(visible=False),
                          title="Simulated network topology (red outline = isolated/blocked; dotted = cut link)", legend=dict(orientation="h"))
        return fig

# ------------------------------------------------------------------ traffic + attack scenarios
@dataclass
class Scenario:
    name: str; kind: str; params: dict; start: float; end: float; state: dict = field(default_factory=dict); active: bool = True

def _ln(rng, mean, sigma, lo=1, hi=10**7): return int(np.clip(rng.lognormal(np.log(mean), sigma), lo, hi))

class TrafficGenerator:
    """Benign traffic mixes (web, DNS, ICMP, file transfer, SSH, UDP). Rates are sessions/second."""
    def __init__(self, sim: NetworkSimulator, rng: np.random.Generator): self.sim, self.rng = sim, rng
    def web(self, t, c, srv="web", heavy=False):
        r = self.rng; n = int(r.integers(1, 5)); port = 443 if r.random() < 0.4 else 80
        ex = [(_ln(r, 450, 0.4, 80, 3000), _ln(r, 9000 if not heavy else 80000, 1.0, 100, 400000), float(r.exponential(0.25))) for _ in range(n)]
        return self.sim.tcp_session(t, c, srv, port, ex, "fin" if r.random() < 0.9 else "rst", "normal", meta={"app": "http"})
    def file_transfer(self, t, c, size=None):
        r = self.rng; up = r.random() < 0.3; size = size or _ln(r, 400_000, 0.8, 20_000, 1_500_000)
        ex = [(60, 50, 0.01), (30, 40, 0.05)] + ([(size, 60, 0.1)] if up else [(40, size, 0.1)])
        return self.sim.tcp_session(t, c, "ftp", 21, ex, "fin", "normal", meta={"app": "ftp-transfer"})
    def ssh(self, t, c):
        r = self.rng; ex = [(21, 21, 0.0), (_ln(r, 900, 0.2), _ln(r, 900, 0.2), 0.01)] + [(int(r.integers(36, 90)), int(r.integers(36, 400)), float(r.exponential(0.6))) for _ in range(int(r.integers(8, 40)))]
        return self.sim.tcp_session(t, c, "ssh", 22, ex, "fin", "normal", meta={"app": "ssh"})
    def ping(self, t, c, dst=None):
        dst = dst or str(self.rng.choice(["web", "dns", "ftp", "ssh"])); return self.sim.icmp_echo(t, c, dst, 56, int(self.rng.integers(1, 5)), 1.0)
    def udp(self, t, c): return self.sim.udp_exchange(t, c, "dns", 53, int(self.rng.integers(30, 80)), int(self.rng.integers(80, 300)), meta={"app": "dns"})
    def session(self, t, c):
        u = self.rng.random()
        if u < 0.45: return self.web(t, c)
        if u < 0.65: return self.sim.dns_query(t, c)
        if u < 0.72: return self.ping(t, c)
        if u < 0.82: return self.file_transfer(t, c)
        if u < 0.90: return self.ssh(t, c)
        return self.udp(t, c)
    def emit_normal(self, sc: Scenario, t0, t1):
        clients = sc.params.get("clients") or [f"c{i}" for i in range(1, 7)]
        for t in sorted(t0 + self.rng.random(self.rng.poisson(sc.params.get("rate", 3.0) * (t1 - t0))) * (t1 - t0)):
            self.session(float(t), str(self.rng.choice(clients)))

class AttackSimulator:
    """Controlled attack-like traffic inside the simulator. Flow features are DERIVED from the packets it generates."""
    def __init__(self, sim: NetworkSimulator, rng: np.random.Generator, tg: TrafficGenerator): self.sim, self.rng, self.tg = sim, rng, tg
    def _rate(self, sc, t):  # intensity scaling + gradual ramp
        ramp = sc.params.get("ramp", 0.0); f = min(1.0, (t - sc.start) / ramp) if ramp > 0 else 1.0
        return sc.params.get("rate", 10.0) * sc.params.get("intensity", 1.0) * max(f, 0.02)
    def _times(self, sc, t0, t1):
        lam = self._rate(sc, (t0 + t1) / 2) * (t1 - t0); n = self.rng.poisson(lam)
        return sorted(float(x) for x in t0 + self.rng.random(n) * (t1 - t0))
    def emit(self, sc: Scenario, t0, t1):
        getattr(self, "_" + sc.kind)(sc, t0, t1)
    def _srcs(self, sc): s = sc.params.get("src", "a1"); return s if isinstance(s, list) else [s]
    def _dos(self, sc, t0, t1):
        dst, r = sc.params.get("dst", "web"), self.rng; srcs = self._srcs(sc)
        for t in self._times(sc, t0, t1):
            s = str(r.choice(srcs))
            if sc.params.get("variant") == "slow":      # slowloris-like: tiny header chunks every ~10 s
                ex = [(5, 0, float(r.uniform(8, 14))) for _ in range(int(r.integers(3, 7)))]
                self.sim.tcp_session(t, s, dst, 80, ex, "none", "DoS", meta={"app": "slow-http"})
            else:
                self.sim.tcp_session(t, s, dst, 80, [(_ln(r, 380, 0.25, 200, 900), int(r.choice([0, 0, 180, 500])), 0.0)], "rst" if r.random() < 0.6 else "none", "DoS", meta={"app": "http-flood"})
    def _ddos(self, sc, t0, t1):
        dst, r = sc.params.get("dst", "web"), self.rng; srcs = self._srcs(sc)
        for t in self._times(sc, t0, t1):
            s = str(r.choice(srcs)); self.sim.tcp_session(t, s, dst, 80, [(int(r.integers(18, 40)), int(r.choice([0, 0, 120])), 0.0)], "rst" if r.random() < 0.5 else "fin", "DDoS", meta={"app": "http-flood"})
    def _portscan(self, sc, t0, t1):
        p = sc.params; r = self.rng; s = self._srcs(sc)[0]; dst = p.get("dst", "web")
        if "order" not in sc.state:
            lo, hi = p.get("port_range", (1, 1024)); sc.state.update(order=list(r.permutation(np.arange(lo, hi + 1))), i=0, t=sc.start)
        t = max(sc.state["t"], t0); gap = p.get("interval", 0.004) / max(p.get("intensity", 1.0), 0.01)
        while t < t1 and sc.state["i"] < len(sc.state["order"]):
            port = int(sc.state["order"][sc.state["i"]]); sc.state["i"] += 1
            self.sim.tcp_session(t, s, dst, port, [], "none", "PortScan", handshake=False, meta={"app": "syn-scan"}); t += float(r.exponential(gap))
        sc.state["t"] = t
        if sc.state["i"] >= len(sc.state["order"]): sc.active = False
    def _ftp_brute(self, sc, t0, t1):
        r = self.rng; dst = sc.params.get("dst", "ftp")
        for t in self._times(sc, t0, t1):
            ex = [(0, 40, 0.01), (14, 33, 0.02), (16, 24, 0.02), (6, 14, 0.01)]
            self.sim.tcp_session(t, self._srcs(sc)[0], dst, 21, ex, "fin", "FTP-Patator", meta={"app": "ftp-login"})
    def _ssh_brute(self, sc, t0, t1):
        r = self.rng; dst = sc.params.get("dst", "ssh")
        for t in self._times(sc, t0, t1):
            ex = [(21, 21, 0.0), (_ln(r, 1000, 0.05, 800, 1200), _ln(r, 1000, 0.05, 800, 1200), 0.02), (48, 60, 0.15), (96, 80, 0.4), (48, 48, 0.2)]
            self.sim.tcp_session(t, self._srcs(sc)[0], dst, 22, ex, "fin", "SSH-Patator", meta={"app": "ssh-login"})
    def _web_attack(self, sc, t0, t1):
        r = self.rng; dst = sc.params.get("dst", "web")
        for t in self._times(sc, t0, t1):
            ex = [(_ln(r, 1800, 0.5, 700, 6000), _ln(r, 350, 0.3, 120, 900), float(r.exponential(0.05))) for _ in range(int(r.integers(2, 6)))]
            self.sim.tcp_session(t, self._srcs(sc)[0], dst, 80, ex, "fin", "WebAttack", meta={"app": "http-suspicious"})
    def _botnet(self, sc, t0, t1):
        r = self.rng; per = sc.params.get("period", 5.0); s = self._srcs(sc)[0]; dst = sc.params.get("dst", "cnc")
        t = sc.state.get("t", sc.start)
        self.sim.dev[s].security_status = "compromised" if self.sim.dev[s].security_status == "normal" else self.sim.dev[s].security_status
        while t < t1:
            if t >= t0:
                ex = [(int(r.normal(120, 6)), int(r.normal(90, 5)), 0.0)]
                if r.random() < 0.1: ex.append((int(r.integers(2000, 9000)), 40, 0.05))   # occasional exfil-like burst
                self.sim.tcp_session(t, s, dst, 8080, ex, "fin", "Bot", meta={"app": "c2-beacon"})
            t += max(0.5, float(r.normal(per, per * 0.1)) / max(sc.params.get("intensity", 1.0), 0.05))
        sc.state["t"] = t
    def _burst(self, sc, t0, t1):   # abnormal connection frequency / legitimate-but-unusual burst (benign-looking sessions)
        s = self._srcs(sc)[0]
        for t in self._times(sc, t0, t1): self.tg.web(t, s, sc.params.get("dst", "web"), heavy=sc.params.get("heavy", False))
    def _unusual(self, sc, t0, t1):  # protocol behaviour absent from the known attack classes
        r = self.rng; s = self._srcs(sc)[0]; var = sc.params.get("variant", "icmp_tunnel")
        if var == "icmp_tunnel":
            t = sc.state.get("t", sc.start); dst = sc.params.get("dst", "cnc")
            while t < t1:
                if t >= t0: self.sim.icmp_echo(t, s, dst, int(r.integers(900, 1300)), 1, label="Unknown")
                t += float(r.normal(0.2, 0.01))
            sc.state["t"] = t
        else:   # xmas-like: FIN|PSH|URG single packets without handshake
            for t in self._times(sc, t0, t1):
                port = int(r.choice([80, 22, 21, 443])); sd, dd = self.sim.dev[s], self.sim.dev[sc.params.get("dst", "web")]
                cid = self.sim.new_cid("TCP", sd.ip, self.sim.eph(), dd.ip, port)
                self.sim.deliver(t, s, dd.id, self.sim.eph(), port, "TCP", FIN | PSH | URG, int(r.integers(0, 20)), cid, "Unknown", {"app": "xmas"})

class ScenarioController:
    """Starts / stops / resets / replays scenarios. Dashboard buttons call these methods directly."""
    KINDS = {"normal": "normal", "dos": "dos", "ddos": "ddos", "portscan": "portscan", "ftp_brute": "ftp_brute", "ssh_brute": "ssh_brute",
             "web_attack": "web_attack", "botnet": "botnet", "burst": "burst", "unusual": "unusual"}
    DEFAULTS = {"normal": dict(rate=3.0), "dos": dict(src="a1", dst="web", rate=120.0), "ddos": dict(src=["a1", "a2", "a3"], dst="web", rate=200.0),
                "portscan": dict(src="a1", dst="web", port_range=(1, 1024), interval=0.004), "ftp_brute": dict(src="a2", dst="ftp", rate=3.0),
                "ssh_brute": dict(src="a2", dst="ssh", rate=2.0), "web_attack": dict(src="a3", dst="web", rate=8.0),
                "botnet": dict(src="c3", dst="cnc", period=5.0), "burst": dict(src="c5", dst="web", rate=30.0),
                "unusual": dict(src="c4", dst="cnc", variant="icmp_tunnel")}
    def __init__(self, sim: NetworkSimulator, seed: int = 42):
        self.sim = sim; self.seed = seed; self.rng = np.random.default_rng(seed + 1)
        self.tg = TrafficGenerator(sim, self.rng); self.atk = AttackSimulator(sim, self.rng, self.tg); self.scenarios: List[Scenario] = []; self.history: List[dict] = []
    def start(self, kind: str, duration: float = 30.0, **params) -> Scenario:
        if kind not in self.KINDS: raise ValueError(f"unknown scenario {kind}")
        p = {**self.DEFAULTS[kind], **params}; sc = Scenario(f"{kind}#{len(self.history)+1}", kind, p, self.sim.clock, self.sim.clock + duration)
        self.scenarios.append(sc); self.history.append(dict(name=sc.name, kind=kind, params=p, start=sc.start, duration=duration)); log.info("start %s %s", sc.name, p); return sc
    def stop(self, name: Optional[str] = None):
        for s in self.scenarios:
            if name is None or s.name == name or s.kind == name: s.active = False
    def tick(self, dt: float = 2.0):
        """Advance the simulated clock by dt, emitting events of every active scenario in [clock, clock+dt)."""
        t0, t1 = self.sim.clock, self.sim.clock + dt
        for s in self.scenarios:
            if not s.active: continue
            a, b = max(t0, s.start), min(t1, s.end)
            if b > a: (self.tg.emit_normal(s, a, b) if s.kind == "normal" else self.atk.emit(s, a, b))
            if s.end <= t1: s.active = False
        self.sim.clock = t1; self.sim.router.expire(t1)
    def run(self, seconds: float, dt: float = 2.0):
        n = int(np.ceil(seconds / dt))
        for _ in range(n): self.tick(dt)
    def reset(self, seed: Optional[int] = None):
        """Reset the whole network (devices, ACLs, clock, capture) and the scenario state with a fresh seed."""
        if seed is not None: self.seed = seed
        self.sim.reset(self.seed); self.rng = np.random.default_rng(self.seed + 1)
        self.tg = TrafficGenerator(self.sim, self.rng); self.atk = AttackSimulator(self.sim, self.rng, self.tg)
        self.scenarios, self.history = [], []
    def replay(self, plan: List[dict], seed: Optional[int] = None, extra: float = 10.0):
        """Deterministically re-run a recorded plan [{kind, params, start, duration}] on a fresh network."""
        self.reset(seed)
        for it in sorted(plan, key=lambda x: x["start"]):
            while self.sim.clock < it["start"]: self.tick(min(2.0, it["start"] - self.sim.clock))
            self.start(it["kind"], it["duration"], **it["params"])
        self.run(max(it["start"] + it["duration"] for it in plan) - self.sim.clock + extra)

# ------------------------------------------------------------------ flow extraction
class _Flow:
    __slots__ = ("key", "src", "dst", "sp", "dp", "proto", "ts", "ln", "fw", "fl", "labels", "devs", "term", "fin_f", "fin_b", "cid", "last")
class FlowFeatureExtractor:
    """CICFlowMeter-style bidirectional flow assembly. Forward = direction of the first packet of the flow.
    Flow ends on RST, on FIN from both sides (after a short linger), on timeout, or on forced flush."""
    LINGER = 2.0
    def __init__(self, cfg: Optional[Config] = None):
        self.cfg = cfg or Config(); self.active: Dict[tuple, _Flow] = {}; self.done: List[dict] = []; self._n = itertools.count(1)
    @staticmethod
    def _key(p: Packet):
        a, b = (p.src_ip, p.sport), (p.dst_ip, p.dport); return (p.proto,) + ((a, b) if a <= b else (b, a))
    def ingest(self, packets: List[Packet]):
        for p in packets:
            k = self._key(p); f = self.active.get(k)
            if f is not None:
                new_conn = p.proto == "TCP" and (p.flags & SYN) and not (p.flags & ACK) and f.term
                if p.ts - f.last > self.cfg.flow_timeout_s or new_conn: self._finish(k)
                f = self.active.get(k)
            if f is None:
                f = _Flow(); f.key, f.src, f.dst, f.sp, f.dp, f.proto = k, p.src_ip, p.dst_ip, p.sport, p.dport, p.proto
                f.ts, f.ln, f.fw, f.fl, f.labels, f.devs = [], [], [], [], [], (p.src_dev, p.dst_dev)
                f.term, f.fin_f, f.fin_b, f.cid, f.last = False, False, False, p.conn_id, p.ts; self.active[k] = f
            fwd = p.src_ip == f.src and p.sport == f.sp
            f.ts.append(p.ts); f.ln.append(p.payload); f.fw.append(fwd); f.fl.append(p.flags); f.labels.append(p.label); f.last = p.ts
            if p.proto == "TCP":
                if p.flags & RST: f.term = True
                if p.flags & FIN: (setattr(f, "fin_f", True) if fwd else setattr(f, "fin_b", True)); f.term = f.term or (f.fin_f and f.fin_b)
    def _finish(self, k):
        f = self.active.pop(k); self.done.append(self._features(f))
    def pop_finished(self, now: float, force: bool = False) -> List[dict]:
        for k, f in list(self.active.items()):
            if force or (f.term and now - f.last >= self.LINGER) or now - f.last > self.cfg.flow_timeout_s: self._finish(k)
        out, self.done = self.done, []; return out
    @staticmethod
    def _stats(a: np.ndarray):
        if a.size == 0: return 0.0, 0.0, 0.0, 0.0
        return float(a.mean()), float(a.std(ddof=1)) if a.size > 1 else 0.0, float(a.max()), float(a.min())
    def _features(self, f: _Flow) -> dict:
        ts = np.asarray(f.ts) * 1e6; ln = np.asarray(f.ln, float); fw = np.asarray(f.fw, bool); fl = np.asarray(f.fl, int)
        o = np.argsort(ts, kind="stable"); ts, ln, fw, fl = ts[o], ln[o], fw[o], fl[o]
        dur = float(ts[-1] - ts[0]); n = len(ts); nf, nb = int(fw.sum()), int((~fw).sum())
        lf, lb = ln[fw], ln[~fw]; iat = np.diff(ts); fi = np.diff(ts[fw]); bi = np.diff(ts[~fw])
        m, s, mx, mn = self._stats(iat); fm, fs, fmx, fmn = self._stats(fi); bm, bs, bmx, bmn = self._stats(bi)
        lfm, lfs, lfx, lfn = self._stats(lf); lbm, lbs, lbx, lbn = self._stats(lb); pm, ps, px, pn = self._stats(ln)
        sec = dur / 1e6
        act, idl, start = [], [], ts[0]
        for i in range(1, n):
            gap = ts[i] - ts[i - 1]
            if gap > self.cfg.activity_timeout_us:
                if ts[i - 1] - start > 0: act.append(ts[i - 1] - start)
                idl.append(gap); start = ts[i]
        if ts[-1] - start > 0: act.append(ts[-1] - start)
        am, as_, ax, an = self._stats(np.asarray(act)); im, is_, ix, in_ = self._stats(np.asarray(idl))
        cnt = lambda bit: int(((fl & bit) > 0).sum())
        row = {
         "Flow Duration": dur, "Total Fwd Packets": nf, "Total Backward Packets": nb, "Total Length of Fwd Packets": float(lf.sum()),
         "Total Length of Bwd Packets": float(lb.sum()), "Fwd Packet Length Max": lfx, "Fwd Packet Length Min": lfn, "Fwd Packet Length Mean": lfm,
         "Fwd Packet Length Std": lfs, "Bwd Packet Length Max": lbx, "Bwd Packet Length Min": lbn, "Bwd Packet Length Mean": lbm, "Bwd Packet Length Std": lbs,
         "Flow Bytes/s": float(ln.sum() / sec) if sec > 0 else 0.0, "Flow Packets/s": n / sec if sec > 0 else 0.0,
         "Flow IAT Mean": m, "Flow IAT Std": s, "Flow IAT Max": mx, "Flow IAT Min": mn,
         "Fwd IAT Total": float(fi.sum()), "Fwd IAT Mean": fm, "Fwd IAT Std": fs, "Fwd IAT Max": fmx, "Fwd IAT Min": fmn,
         "Bwd IAT Total": float(bi.sum()), "Bwd IAT Mean": bm, "Bwd IAT Std": bs, "Bwd IAT Max": bmx, "Bwd IAT Min": bmn,
         "Fwd PSH Flags": int(((fl[fw] & PSH) > 0).sum()), "Fwd Packets/s": nf / sec if sec > 0 else 0.0, "Bwd Packets/s": nb / sec if sec > 0 else 0.0,
         "Min Packet Length": pn, "Max Packet Length": px, "Packet Length Mean": pm, "Packet Length Std": ps, "Packet Length Variance": ps ** 2,
         "FIN Flag Count": cnt(FIN), "SYN Flag Count": cnt(SYN), "RST Flag Count": cnt(RST), "PSH Flag Count": cnt(PSH), "ACK Flag Count": cnt(ACK), "URG Flag Count": cnt(URG),
         "Down/Up Ratio": float(nb // nf) if nf else 0.0, "Average Packet Size": float(ln.mean()), "Avg Fwd Segment Size": lfm, "Avg Bwd Segment Size": lbm,
         "Active Mean": am, "Active Std": as_, "Active Max": ax, "Active Min": an, "Idle Mean": im, "Idle Std": is_, "Idle Max": ix, "Idle Min": in_}
        lab = max(set(f.labels), key=f.labels.count)
        row.update(flow_id=f"F{next(self._n)}", src_ip=f.src, dst_ip=f.dst, src_port=f.sp, dst_port=f.dp, proto=f.proto,
                   start_ts=float(ts[0] / 1e6), end_ts=float(ts[-1] / 1e6), true_label=lab, conn_id=f.cid)
        return row

Overwriting /content/ids_src/sim.py


In [4]:
%%writefile /content/ids_src/data.py
"""data.py - CIC-IDS2017 loading, inspection, cleaning and leakage-safe chronological splitting."""
from __future__ import annotations
import glob, os
from typing import Dict, Optional, Tuple
import numpy as np
import pandas as pd
from core import Config, FeatureSchema, get_logger

log = get_logger("ids.data")

def normalize_label(s: str) -> str:
    """Map raw CIC labels to the merged taxonomy used by this project (mapping is documented in the report)."""
    t = str(s).strip().lower()
    if "web attack" in t: return "WebAttack"
    if t.startswith("dos"): return "DoS"
    if t == "ddos": return "DDoS"
    if t == "portscan": return "PortScan"
    if "ftp" in t: return "FTP-Patator"
    if "ssh" in t: return "SSH-Patator"
    if t == "bot": return "Bot"
    if t == "benign": return "BENIGN"
    if "infiltration" in t: return "Infiltration"
    if "heartbleed" in t: return "Heartbleed"
    return str(s).strip()

class DatasetProcessor:
    """Loads TrafficLabelling CSVs, cleans them with the SAME rules as the simulator path, and splits chronologically."""
    def __init__(self, cfg: Config): self.cfg = cfg; self.report: Dict[str, object] = {}
    def find_files(self) -> list:
        fs = sorted(glob.glob(os.path.join(self.cfg.data_dir, "**", "*.csv"), recursive=True))
        # Explicitly filter only valid CIC-IDS2017 dataset files and ignore other non-dataset CSV files
        fs = [f for f in fs if ("WorkingHours" in os.path.basename(f) or f.endswith("pcap_ISCX.csv")) and "Maths" not in os.path.basename(f)]
        if not fs:
            raise FileNotFoundError(f"No CIC-IDS2017 CSV files under {self.cfg.data_dir}. Place the TrafficLabelling CSVs there.")
        return fs
    def inspect_file(self, path: str, nrows: int = 5000) -> pd.DataFrame:
        raw = pd.read_csv(path, nrows=nrows, encoding="latin1"); raw.columns = [c.strip() for c in raw.columns]
        return raw
    def load(self) -> pd.DataFrame:
        """Chunked, downcast load. Keeps schema features + Timestamp + Label. Caps rows per class (time-ordered systematic sampling)."""
        want = set(FeatureSchema.names) | {"Label", "Timestamp"}; parts, raw_n, order = [], 0, 0
        for f in self.find_files():
            log.info("loading %s", os.path.basename(f))
            for ch in pd.read_csv(f, usecols=lambda c: c.strip() in want, chunksize=250_000, encoding="latin1", low_memory=False):
                ch.columns = [c.strip() for c in ch.columns]; ch = ch.loc[:, ~ch.columns.duplicated()]
                raw_n += len(ch); ch["Label"] = ch["Label"].map(normalize_label)
                for c in FeatureSchema.names: ch[c] = pd.to_numeric(ch[c], errors="coerce").astype("float64")
                ch["Timestamp"] = self._parse_ts(ch["Timestamp"])
                ch["_order"] = np.arange(order, order + len(ch)); order += len(ch); parts.append(ch)
        df = pd.concat(parts, ignore_index=True); del parts
        missing = [c for c in FeatureSchema.names if c not in df.columns]
        if missing: raise ValueError(f"Dataset lacks schema features: {missing}")
        self.report["raw_rows"] = raw_n; self.report["raw_label_counts"] = df["Label"].value_counts().to_dict()
        return self.clean(df)
    @staticmethod
    def _parse_ts(s: pd.Series) -> pd.Series:
        """Parse CIC timestamps (month-first). Files record 12-hour clock times without AM/PM; captures ran ~08:00-17:00, so hours 1-7 are PM (+12h). Heuristic, documented."""
        t = pd.to_datetime(s, errors="coerce")
        bad = t.isna() & s.notna()
        if bad.any(): t[bad] = pd.to_datetime(s[bad], errors="coerce", format="mixed")
        pm = t.notna() & (t.dt.hour < 8); t[pm] = t[pm] + pd.Timedelta(hours=12); return t
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        r = self.report; n0 = len(df)
        x = FeatureSchema.apply_zero_duration_rule(df[FeatureSchema.names]); df[FeatureSchema.names] = x
        arr = df[FeatureSchema.names].replace([np.inf, -np.inf], np.nan)
        r["nan_or_inf_cells"] = int(arr.isna().sum().sum()); r["negative_cells"] = int((arr < 0).sum().sum())
        r["constant_features"] = [c for c in FeatureSchema.names if df[c].nunique(dropna=True) <= 1]
        ok = FeatureSchema.valid_mask(arr); r["rows_dropped_invalid"] = int((~ok).sum()); df = df[ok.values].copy()
        n1 = len(df); df = df.drop_duplicates(subset=FeatureSchema.names + ["Label"]); r["duplicates_removed"] = n1 - len(df)
        if df["Timestamp"].isna().mean() > 0.2: log.warning("Timestamps unparsable -> falling back to file order"); df["Timestamp"] = pd.to_datetime(df["_order"], unit="s")
        df["Timestamp"] = df["Timestamp"].fillna(pd.Timestamp("1970-01-01"))
        df[FeatureSchema.names] = df[FeatureSchema.names].astype("float32")
        cnt = df["Label"].value_counts(); keep = cnt[cnt >= self.cfg.min_class_rows].index
        r["excluded_rare_classes"] = {k: int(v) for k, v in cnt.items() if k not in keep}
        self.rare = df[~df["Label"].isin(keep)].copy(); df = df[df["Label"].isin(keep)]
        df = self._cap(df); r["clean_rows"] = len(df); r["clean_label_counts"] = df["Label"].value_counts().to_dict()
        log.info("clean rows %d (from %d)", len(df), n0); return df.reset_index(drop=True)
    def _cap(self, df):
        out = []
        for lab, g in df.groupby("Label"):
            g = g.sort_values(["Timestamp", "_order"])
            if len(g) > self.cfg.max_rows_per_class: g = g.iloc[np.linspace(0, len(g) - 1, self.cfg.max_rows_per_class).astype(int)]
            out.append(g)
        return pd.concat(out)
    def split(self, df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        """Per-class chronological blocks train/val/test with a gap buffer between blocks (no random row-level split)."""
        a, b, _ = self.cfg.split; gap = self.cfg.split_gap; parts = {"train": [], "val": [], "test": []}
        for lab, g in df.groupby("Label"):
            g = g.sort_values(["Timestamp", "_order"]); n = len(g); i, j, gp = int(n * a), int(n * (a + b)), max(1, int(n * gap))
            parts["train"].append(g.iloc[: i - gp]); parts["val"].append(g.iloc[i: j - gp]); parts["test"].append(g.iloc[j:])
        out = {k: pd.concat(v).sort_values(["Timestamp", "_order"]).reset_index(drop=True) for k, v in parts.items()}
        self.leakage_check(out); return out
    @staticmethod
    def leakage_check(sp: Dict[str, pd.DataFrame]):
        # row-id disjointness and strict time ordering per class
        ids = [set(sp[k]["_order"]) for k in ("train", "val", "test")]
        assert not (ids[0] & ids[1]) and not (ids[0] & ids[2]) and not (ids[1] & ids[2]), "row overlap between splits"
        for lab in sp["train"]["Label"].unique():
            t = {k: sp[k].loc[sp[k]["Label"] == lab, "Timestamp"] for k in sp}
            assert t["train"].max() <= t["val"].min() and t["val"].max() <= t["test"].min(), f"temporal leakage in class {lab}"
        log.info("leakage checks passed (disjoint rows, per-class chronological order)")

Overwriting /content/ids_src/data.py


In [5]:
%%writefile /content/ids_src/models.py
"""models.py - RF, XGBoost, Isolation Forest, Autoencoder, calibration and the unified HybridIntrusionDetector."""
from __future__ import annotations
import itertools, time
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from core import Config, FeatureSchema, Preprocessor, get_logger

log = get_logger("ids.models")
BENIGN = "BENIGN"; UNKNOWN = "UNKNOWN_ANOMALY"

# ------------------------------------------------------------------ base detectors
class RandomForestDetector:
    def __init__(self, seed=42, n=80, **kw):
        self.m = RandomForestClassifier(n_estimators=n, max_depth=20, min_samples_leaf=5, n_jobs=-1, random_state=seed, class_weight="balanced_subsample", **kw)
    def fit(self, X, y): self.m.fit(X, y); return self
    def predict_proba(self, X): return self.m.predict_proba(X)
    @property
    def importances(self): return self.m.feature_importances_

class XGBoostDetector:
    def __init__(self, n_classes: int, seed=42):
        import xgboost as xgb
        self.m = xgb.XGBClassifier(objective="multi:softprob", n_estimators=300, max_depth=8, learning_rate=0.1, subsample=0.8,
                                   colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=seed, early_stopping_rounds=15, eval_metric="mlogloss")
    def fit(self, X, y, Xv, yv):
        self.m.fit(X, y, sample_weight=compute_sample_weight("balanced", y), eval_set=[(Xv, yv)], verbose=False); return self
    def predict_proba(self, X): return self.m.predict_proba(X)

class IsolationForestDetector:
    """Unsupervised; trained on benign TRAIN flows only. score = -score_samples (higher = more anomalous)."""
    def __init__(self, seed=42): self.m = IsolationForest(n_estimators=150, max_samples=512, contamination="auto", n_jobs=-1, random_state=seed)
    def fit(self, Xb): self.m.fit(Xb); return self
    def score(self, X): return -self.m.score_samples(X)

class AutoencoderDetector:
    """Dense autoencoder (d->32->16->8->16->32->d) trained on benign flows; reconstruction MSE = anomaly score."""
    def __init__(self, d: int, seed=42): self.d, self.seed, self.m = d, seed, None
    def _build(self):
        import tensorflow as tf
        tf.keras.utils.set_random_seed(self.seed); L = tf.keras.layers
        m = tf.keras.Sequential([L.Input((self.d,)), L.Dense(32, activation="relu"), L.Dense(16, activation="relu"), L.Dense(8, activation="relu"),
                                 L.Dense(16, activation="relu"), L.Dense(32, activation="relu"), L.Dense(self.d)])
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse"); return m
    def fit(self, Xb, Xbv, epochs=40):
        import tensorflow as tf
        self.m = self._build()
        self.history = self.m.fit(Xb, Xb, validation_data=(Xbv, Xbv), epochs=epochs, batch_size=512, verbose=0,
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_loss")]).history; return self
    def score(self, X): R = self.m.predict(X, batch_size=4096, verbose=0); return ((X - R) ** 2).mean(1)

# ------------------------------------------------------------------ calibration helpers
class BenignCDF:
    """Maps a raw anomaly score to its quantile among BENIGN calibration scores (0..1), putting IF and AE on one scale."""
    def fit(self, benign_scores): self.s = np.sort(np.asarray(benign_scores)); return self
    def __call__(self, x): return np.searchsorted(self.s, np.asarray(x), side="right") / len(self.s)

class ThresholdCalibrator:
    """All thresholds come from VALIDATION data only."""
    @staticmethod
    def benign_quantile(scores_benign: np.ndarray, target_fpr: float) -> float: return float(np.quantile(scores_benign, 1 - target_fpr))
    @staticmethod
    def tpr_at_fpr(score: np.ndarray, y_att: np.ndarray, fpr: float) -> Tuple[float, float]:
        thr = ThresholdCalibrator.benign_quantile(score[y_att == 0], fpr); return float((score[y_att == 1] > thr).mean()), thr
    @staticmethod
    def tune_class_bias(P: np.ndarray, y: np.ndarray, grid=(0.5, 0.75, 1, 1.5, 2, 3, 5)) -> np.ndarray:
        """Per-class probability multipliers (coordinate ascent on validation macro-F1)."""
        b = np.ones(P.shape[1]); best = f1_score(y, (P * b).argmax(1), average="macro")
        for _ in range(2):
            for k in range(P.shape[1]):
                for g in grid:
                    t = b.copy(); t[k] = g; f = f1_score(y, (P * t).argmax(1), average="macro")
                    if f > best + 1e-6: best, b = f, t
        return b

# ------------------------------------------------------------------ unified hybrid
class HybridIntrusionDetector:
    """ONE inference object: shared preprocessing -> RF+XGB ensemble -> calibrated S; IF + AE -> benign-CDF I, A -> risk R -> 3-level decision.
    R is chosen on validation among: weighted sum (alpha*S+beta*I+gamma*A), logistic blend, and max-rule."""
    def __init__(self, cfg: Config): self.cfg = cfg; self.classes: List[str] = []; self.cfg_dict: Dict = {}
    # ---- raw component outputs
    def components(self, X: np.ndarray) -> Dict[str, np.ndarray]:
        prf, pxg = self.rf.predict_proba(X), self.xgb.predict_proba(X); ir, ar = self.ifo.score(X), self.ae.score(X)
        P = self.w_rf * prf + (1 - self.w_rf) * pxg; P = P * self.class_bias; P = P / P.sum(1, keepdims=True)
        att = 1 - P[:, self.bi]; S = self.iso.predict(att)
        return dict(prf=prf, pxg=pxg, P=P, S=S, I=self.cdf_if(ir), A=self.cdf_ae(ar), I_raw=ir, A_raw=ar)
    def risk(self, S, I, A) -> np.ndarray:
        f = self.combiner
        if f["kind"] == "weighted": return f["alpha"] * S + f["beta"] * I + f["gamma"] * A
        if f["kind"] == "blend": return f["lr"].predict_proba(np.c_[S, I, A])[:, 1]
        return np.maximum(S, np.minimum(I, A))
    # ---- fitting on validation (val1: calibration, val2: combiner/thresholds)
    def fit_calibration(self, X1, y1, X2, y2):
        c = self.components_uncal(X1); self.class_bias = np.ones(len(self.classes)); self.w_rf = 0.5
        best = (-1, 0.5)
        for w in np.linspace(0, 1, 11):
            f = f1_score(y1, (w * c["prf"] + (1 - w) * c["pxg"]).argmax(1), average="macro"); best = max(best, (f, w))
        self.w_rf = float(best[1]); P = self.w_rf * c["prf"] + (1 - self.w_rf) * c["pxg"]; self.class_bias = ThresholdCalibrator.tune_class_bias(P, y1)
        ya = (y1 != self.bi).astype(int); Pn = P * self.class_bias; Pn /= Pn.sum(1, keepdims=True)
        self.iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(1 - Pn[:, self.bi], ya)
        self.cdf_if = BenignCDF().fit(self.ifo.score(X1[y1 == self.bi])); self.cdf_ae = BenignCDF().fit(self.ae.score(X1[y1 == self.bi]))
        c2 = self.components(X2); ya2 = (y2 != self.bi).astype(int); S, I, A = c2["S"], c2["I"], c2["A"]; fpr = self.cfg.target_fpr_review
        cands = {}
        best_w = (-1, None)
        for a, b in itertools.product(np.arange(0, 1.01, 0.1), repeat=2):
            if a + b > 1.0001: continue
            t, _ = ThresholdCalibrator.tpr_at_fpr(a * S + b * I + (1 - a - b) * A, ya2, fpr)
            if t > best_w[0] + 1e-9: best_w = (t, dict(kind="weighted", alpha=float(a), beta=float(b), gamma=float(1 - a - b)))
        cands["weighted"] = best_w
        Z = np.c_[S, I, A]; lr = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced")
        cv = cross_val_predict(lr, Z, ya2, cv=StratifiedKFold(5, shuffle=True, random_state=self.cfg.seed), method="predict_proba")[:, 1]   # out-of-fold
        cands["blend"] = (ThresholdCalibrator.tpr_at_fpr(cv, ya2, fpr)[0], dict(kind="blend", lr=lr.fit(Z, ya2)))
        cands["maxrule"] = (ThresholdCalibrator.tpr_at_fpr(np.maximum(S, np.minimum(I, A)), ya2, fpr)[0], dict(kind="maxrule"))
        self.combiner_scores = {k: float(v[0]) for k, v in cands.items()}; k = max(cands, key=lambda z: cands[z][0]); self.combiner = cands[k][1]
        R = self.risk(S, I, A); rb = R[ya2 == 0]
        self.tau_review = ThresholdCalibrator.benign_quantile(rb, self.cfg.target_fpr_review); self.tau_high = ThresholdCalibrator.benign_quantile(rb, self.cfg.target_fpr_high)
        self.s_thr = max(0.5, ThresholdCalibrator.benign_quantile(S[ya2 == 0], 0.01)); self.sig_q = self.cfg.signal_quantile
        self.cfg_dict = dict(selected=k, combiner_val_tpr=self.combiner_scores, w_rf=self.w_rf, class_bias=self.class_bias.tolist(), tau_review=self.tau_review,
                             tau_high=self.tau_high, s_thr=self.s_thr, signal_quantile=self.sig_q, combiner={a: b for a, b in self.combiner.items() if a != "lr"})
        log.info("hybrid calibrated: %s", self.cfg_dict); return self
    def components_uncal(self, X):
        return dict(prf=self.rf.predict_proba(X), pxg=self.xgb.predict_proba(X))
    # ---- inference ---------------------------------------------------------------------
    def predict_arrays(self, X: np.ndarray) -> Dict[str, np.ndarray]:
        c = self.components(X); R = self.risk(c["S"], c["I"], c["A"])
        sup, ifs, aes = c["S"] >= self.s_thr, c["I"] >= self.sig_q, c["A"] >= self.sig_q; n = sup.astype(int) + ifs + aes
        high = (R >= self.tau_high) & (n >= 2) & sup
        review = (R >= self.tau_review) | (~sup & ifs & aes)
        status = np.where(high, "HIGH", np.where(review, "REVIEW", "NORMAL"))
        att = np.array([i for i in range(len(self.classes)) if i != self.bi]); top = att[c["P"][:, att].argmax(1)]
        cat = np.where(status == "NORMAL", BENIGN, np.where(sup, np.array(self.classes)[top], UNKNOWN))
        c.update(R=R, status=status, category=cat, sup=sup, ifs=ifs, aes=aes, n_signals=n); return c
    def predict(self, flows: pd.DataFrame) -> pd.DataFrame:
        """Validate schema -> shared preprocessing -> unified result per flow (adds columns to a copy of the metadata)."""
        Xdf = FeatureSchema.clean(flows)
        if not FeatureSchema.valid_mask(Xdf).all(): raise ValueError("Invalid (NaN/Inf/negative) feature values in flows")
        if self.pre.schema_hash != FeatureSchema.version_hash(): raise ValueError("Schema hash mismatch between model and runtime")
        c = self.predict_arrays(self.pre.transform(Xdf)); out = flows.drop(columns=FeatureSchema.names, errors="ignore").copy()
        att = [i for i in range(len(self.classes)) if i != self.bi]
        out["status"], out["category"], out["risk"] = c["status"], c["category"], c["R"]
        out["S_sup"], out["I_iforest"], out["A_ae"] = c["S"], c["I"], c["A"]; out["ae_recon_error"], out["if_raw_score"] = c["A_raw"], c["I_raw"]
        out["rf_attack_p"], out["xgb_attack_p"] = 1 - c["prf"][:, self.bi], 1 - c["pxg"][:, self.bi]
        out["rf_class"] = np.array(self.classes)[c["prf"].argmax(1)]; out["xgb_class"] = np.array(self.classes)[c["pxg"].argmax(1)]
        out["signals"] = c["n_signals"]
        out["agreement"] = [f"RF={a}, XGB={b}, IF={'anomalous' if i else 'normal'}, AE={'elevated' if e else 'normal'}" + (" [UNANIMOUS]" if (a != BENIGN) == (b != BENIGN) == bool(i) == bool(e) else " [DISAGREE]")
                            for a, b, i, e in zip(out["rf_class"], out["xgb_class"], c["ifs"], c["aes"])]
        out["action"] = [self.recommend(s, k) for s, k in zip(out["status"], out["category"])]
        for j, k in enumerate(self.classes): out[f"P_{k}"] = c["P"][:, j]
        return out
    @staticmethod
    def recommend(status: str, cat: str) -> str:
        if status == "NORMAL": return "Allow; continue monitoring"
        if status == "REVIEW": return "Warn + raise monitoring + preserve evidence; administrator review (no automatic blocking)"
        return {"PortScan": "Block source IP (ACL) after review; restrict scanned ports", "DoS": "Rate-limit / block source IP; verify service health", "DDoS": "Block participating sources; upstream filtering",
                "FTP-Patator": "Block source on port 21; enforce lockout", "SSH-Patator": "Block source on port 22; enforce key-only auth", "Bot": "Isolate host; inspect C2 destination",
                "WebAttack": "Block source / WAF rule; inspect requests"}.get(cat, "Block or isolate after administrator confirmation")
    # ---- explainability ----------------------------------------------------------------
    def fit_explainer(self, Xtrain_benign: np.ndarray):
        self.base_mu = Xtrain_benign.mean(0); self.base_sd = Xtrain_benign.std(0) + 1e-6; self.imp = self.rf.importances; return self
    def explain(self, flow_row: pd.Series, k: int = 5) -> List[str]:
        """Importance-weighted deviation from the benign baseline (always available). SHAP is used separately when installed."""
        x = self.pre.transform(FeatureSchema.clean(pd.DataFrame([flow_row[FeatureSchema.names].astype(float)])))[0]
        z = (x - self.base_mu) / self.base_sd; sc = np.abs(z) * (self.imp / self.imp.max()); idx = np.argsort(-sc)[:k]
        return [f"{'High' if z[i] > 0 else 'Low'} {FeatureSchema.names[i]} (benign z={z[i]:+.1f}, RF importance={self.imp[i]:.3f})" for i in idx]
    def shap_top(self, X: np.ndarray, cls: str, k: int = 5):
        try:
            import shap; ex = shap.TreeExplainer(self.xgb.m); sv = ex.shap_values(X); sv = np.asarray(sv)
            j = self.classes.index(cls); v = sv[:, :, j] if sv.ndim == 3 and sv.shape[2] == len(self.classes) else sv[j]
            return [[(FeatureSchema.names[i], float(r[i])) for i in np.argsort(-np.abs(r))[:k]] for r in v]
        except Exception as e: log.warning("SHAP unavailable: %s", e); return None
    # ---- persistence ---------------------------------------------------------------------
    def save(self, path: str):
        xm, aem = self.xgb.m, self.ae.m; v = (getattr(self, "val1", None), getattr(self, "val2", None)); self.xgb.m = None; self.ae.m = None; self.val1 = self.val2 = None
        try: joblib.dump(self, path + "/hybrid.joblib", compress=3)
        finally: self.xgb.m, self.ae.m = xm, aem; self.val1, self.val2 = v
        xm.save_model(path + "/xgb.json"); aem.save(path + "/ae.keras")
    @staticmethod
    def load(path: str) -> "HybridIntrusionDetector":
        import xgboost as xgb, tensorflow as tf
        h = joblib.load(path + "/hybrid.joblib"); h.xgb.m = xgb.XGBClassifier(); h.xgb.m.load_model(path + "/xgb.json"); h.ae.m = tf.keras.models.load_model(path + "/ae.keras")
        if h.pre.schema_hash != FeatureSchema.version_hash(): raise ValueError("Saved model was trained with a different FeatureSchema")
        return h

class ModelTrainer:
    """Trains RF, XGB, IF, AE on TRAIN only (preprocessing fitted on TRAIN only); calibrates on VALIDATION only."""
    def __init__(self, cfg: Config): self.cfg = cfg; self.timings: Dict[str, float] = {}
    def _cap(self, df: pd.DataFrame, n: int, rng) -> pd.DataFrame:
        return pd.concat([g.sample(min(len(g), n), random_state=self.cfg.seed) for _, g in df.groupby("Label")])
    def compare_imbalance(self, tr: pd.DataFrame, va: pd.DataFrame, pre: Preprocessor, classes: List[str], n_small=80_000) -> pd.DataFrame:
        """Validation macro-F1 of 3 strategies with a small RF: class weights / capped undersampling / random oversampling."""
        rng = np.random.default_rng(self.cfg.seed); y = lambda d: d["Label"].map({c: i for i, c in enumerate(classes)}).values
        Xv, yv = pre.transform(va[FeatureSchema.names]), y(va); res = {}
        base = self._cap(tr, n_small, rng); med = int(base["Label"].value_counts().median())
        for name, d, cw in [("none", base, None), ("class_weight", base, "balanced_subsample"),
                            ("undersample(cap 5k)+none", self._cap(tr, 5_000, rng), None),
                            ("oversample(to median)", pd.concat([g.sample(med, replace=len(g) < med, random_state=1) for _, g in base.groupby("Label")]), None)]:
            m = RandomForestClassifier(n_estimators=60, max_depth=20, n_jobs=-1, random_state=self.cfg.seed, class_weight=cw).fit(pre.transform(d[FeatureSchema.names]), y(d))
            res[name] = dict(macro_f1=f1_score(yv, m.predict(Xv), average="macro"))
        return pd.DataFrame(res).T
    def train(self, sp: Dict[str, pd.DataFrame]) -> HybridIntrusionDetector:
        cfg = self.cfg; rng = np.random.default_rng(cfg.seed); tr, va = sp["train"], sp["val"]
        classes = sorted(tr["Label"].unique(), key=lambda c: (c != BENIGN, c)); h = HybridIntrusionDetector(cfg); h.classes, h.bi = classes, classes.index(BENIGN)
        enc = {c: i for i, c in enumerate(classes)}; h.pre = Preprocessor().fit(tr[FeatureSchema.names])
        trc = self._cap(tr, cfg.train_cap_per_class, rng); Xt, yt = h.pre.transform(trc[FeatureSchema.names]), trc["Label"].map(enc).values
        # validation halves per class, chronological: val1 = calibration, val2 = combiner/thresholds
        v1, v2 = [], []
        for _, g in va.groupby("Label"): g = g.sort_values("Timestamp"); k = len(g) // 2; v1.append(g.iloc[:k]); v2.append(g.iloc[k:])
        v1, v2 = pd.concat(v1), pd.concat(v2); X1, y1 = h.pre.transform(v1[FeatureSchema.names]), v1["Label"].map(enc).values; X2, y2 = h.pre.transform(v2[FeatureSchema.names]), v2["Label"].map(enc).values
        t = time.time(); h.rf = RandomForestDetector(cfg.seed).fit(Xt, yt); self.timings["rf"] = time.time() - t
        t = time.time(); h.xgb = XGBoostDetector(len(classes), cfg.seed).fit(Xt, yt, X1, y1); self.timings["xgb"] = time.time() - t
        Xb = h.pre.transform(tr.loc[tr["Label"] == BENIGN, FeatureSchema.names].sample(min(300_000, (tr["Label"] == BENIGN).sum()), random_state=cfg.seed))
        t = time.time(); h.ifo = IsolationForestDetector(cfg.seed).fit(Xb[:200_000]); self.timings["iforest"] = time.time() - t
        t = time.time(); h.ae = AutoencoderDetector(Xb.shape[1], cfg.seed).fit(Xb, X1[y1 == h.bi]); self.timings["autoencoder"] = time.time() - t
        h.fit_calibration(X1, y1, X2, y2); h.fit_explainer(Xb); h.val1, h.val2 = v1, v2; return h

Overwriting /content/ids_src/models.py


In [6]:
%%writefile /content/ids_src/ids.py
"""ids.py - temporal detection, alert management, false-positive management and the response engine."""
from __future__ import annotations
import itertools
from collections import deque, defaultdict
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
from core import Config, get_logger
from sim import NetworkSimulator
from models import UNKNOWN

log = get_logger("ids.ids")

class TemporalDetectionEngine:
    """Per-source-host rolling windows. Computes port diversity / connection frequency (host-level features that are NOT in the flow
    schema), EWMA risk, consecutive-suspicious counts and a per-host flow-rate baseline. May escalate REVIEW -> HIGH, never downgrade."""
    def __init__(self, cfg: Config, tau_review: float, ewma_alpha: float = 0.3):
        self.cfg, self.tau, self.a = cfg, tau_review, ewma_alpha; self.h: Dict[str, deque] = defaultdict(lambda: deque(maxlen=5000))
        self.ewma: Dict[str, float] = {}; self.consec: Dict[str, int] = defaultdict(int); self.base: Dict[str, list] = defaultdict(lambda: [0, 0.0, 0.0])
    def observe(self, row: dict) -> dict:
        src, t, R, st = row["src_ip"], row["start_ts"], row["risk"], row["status"]
        self.h[src].append((t, R, st, row["dst_ip"], row["dst_port"])); self.ewma[src] = self.a * R + (1 - self.a) * self.ewma.get(src, R)
        self.consec[src] = self.consec[src] + 1 if st != "NORMAL" else 0
        feats = {}
        for w in self.cfg.temporal_windows_s:
            rec = [r for r in self.h[src] if t - r[0] <= w]; feats[f"w{int(w)}"] = dict(flows=len(rec), conn_per_s=len(rec) / w, dst_ports=len({r[4] for r in rec}), dst_hosts=len({r[3] for r in rec}),
                                                                                           mean_risk=float(np.mean([r[1] for r in rec])), suspicious=sum(r[2] != "NORMAL" for r in rec))
        rate = feats[f"w{int(self.cfg.temporal_windows_s[0])}"]["conn_per_s"]; n, m, M2 = self.base[src]
        z = (rate - m) / (np.sqrt(M2 / n) + 1e-6) if n > 5 else 0.0
        if st == "NORMAL":   # baseline updated on normal flows only (Welford)
            n += 1; d = rate - m; m += d / n; M2 += d * (rate - m); self.base[src] = [n, m, M2]
        w0 = feats[f"w{int(self.cfg.temporal_windows_s[0])}"]
        support = (self.consec[src] >= self.cfg.temporal_consecutive and self.ewma[src] >= self.tau) or (w0["dst_ports"] >= 20 and st != "NORMAL")
        out = dict(status=st, temporal_support=bool(support), ewma_risk=float(self.ewma[src]), consecutive=int(self.consec[src]), rate_z=float(z),
                   dst_port_diversity_10s=w0["dst_ports"], conn_per_s_10s=w0["conn_per_s"], windows=feats, escalated=False)
        if st == "REVIEW" and support and row["category"] != UNKNOWN: out["status"], out["escalated"] = "HIGH", True
        return out

class FalsePositiveManager:
    """Whitelisting (alters severity/response eligibility, never disables logging) and administrator feedback (never auto-retrains)."""
    def __init__(self): self.whitelist: Dict[str, str] = {}; self.feedback: List[dict] = []
    def add_whitelist(self, ip: str, reason: str = ""): self.whitelist[ip] = reason
    def remove_whitelist(self, ip: str): self.whitelist.pop(ip, None)
    def apply(self, src_ip: str, status: str) -> dict:
        if src_ip in self.whitelist and status != "NORMAL":
            return dict(status="REVIEW" if status == "HIGH" else status, severity_note=f"whitelisted ({self.whitelist[src_ip]}): downgraded, auto-response disabled", auto_ok=False)
        return dict(status=status, severity_note="", auto_ok=True)
    def record(self, alert_id: int, verdict: str, note: str = "", category: str = ""):
        assert verdict in ("confirmed", "false_positive", "investigating"); self.feedback.append(dict(alert_id=alert_id, verdict=verdict, note=note, category=category))
    def summary(self) -> pd.DataFrame:
        return pd.DataFrame(self.feedback) if self.feedback else pd.DataFrame(columns=["alert_id", "verdict", "note", "category"])
    def export_for_review(self, path: str): self.summary().to_csv(path, index=False)   # for MANUAL retraining review only

class AlertManager:
    """Aggregates flow decisions into alerts (key = src, category, dst). All fields originate from model outputs/features."""
    def __init__(self, merge_s: float = 30.0): self.alerts: Dict[int, dict] = {}; self._id = itertools.count(1); self.merge_s = merge_s; self.open: Dict[tuple, int] = {}
    def ingest(self, row: dict, status: str, evidence: List[str], note: str = "") -> Optional[int]:
        if status == "NORMAL": return None
        key = (row["src_ip"], row["category"], row["dst_ip"]); aid = self.open.get(key); sev = "CRITICAL" if status == "HIGH" else "WARNING"
        if aid is not None and row["start_ts"] - self.alerts[aid]["last_time"] <= self.merge_s:
            a = self.alerts[aid]; a["count"] += 1; a["last_time"] = row["start_ts"]
            if row["risk"] > a["risk"]: a.update(risk=row["risk"], evidence=evidence, models=self._m(row))
            if sev == "CRITICAL" and a["severity"] != "CRITICAL": a.update(severity="CRITICAL", action=row["action"])
            return aid
        aid = next(self._id); self.open[key] = aid
        self.alerts[aid] = dict(id=aid, severity=sev, src_ip=row["src_ip"], dst_ip=row["dst_ip"], category=row["category"], status_level=status, risk=row["risk"], count=1,
            first_time=row["start_ts"], last_time=row["start_ts"], evidence=evidence, models=self._m(row), agreement=row["agreement"], action=row["action"],
            investigation="new", note=note, true_label=row.get("true_label", ""))
        return aid
    @staticmethod
    def _m(r): return {k: r[k] for k in ("rf_class", "xgb_class", "rf_attack_p", "xgb_attack_p", "I_iforest", "A_ae", "ae_recon_error", "S_sup", "signals") if k in r}
    def set_status(self, aid: int, st: str): self.alerts[aid]["investigation"] = st
    def table(self) -> pd.DataFrame:
        return pd.DataFrame(self.alerts.values()) if self.alerts else pd.DataFrame()

class ResponseEngine:
    """Maps IDS decisions to simulated controls on the router: ACL/IP block, port restriction, host isolation, expiry, verification, incident log."""
    def __init__(self, sim: NetworkSimulator, cfg: Config): self.sim, self.cfg = sim, cfg; self.incidents: List[dict] = []; self.rules: Dict[int, dict] = {}
    def _log(self, action, target, outcome, **kw): self.incidents.append(dict(time=self.sim.clock, action=action, target=target, outcome=outcome, **kw))
    def _dev(self, ip): return self.sim.by_ip.get(ip)
    def block_ip(self, ip: str, ttl: Optional[float] = None, comment: str = "", alert_id=None, auto=False) -> int:
        r = self.sim.router.add_rule("deny", now=self.sim.clock, ttl=ttl if ttl is not None else self.cfg.rule_ttl_s, src=ip, hard=True, comment=comment or "IDS block")
        self.rules[r.rule_id] = dict(kind="block_ip", target=ip); self._log("block_ip", ip, "applied", rule_id=r.rule_id, alert_id=alert_id, auto=auto); return r.rule_id
    def restrict_port(self, dst_ip: str, port: int, proto: str = "TCP", src: str = "any", ttl: Optional[float] = None, alert_id=None) -> int:
        r = self.sim.router.add_rule("deny", now=self.sim.clock, ttl=ttl, src=src, dst=dst_ip, proto=proto, dport=port, hard=True, comment=f"restrict {proto}/{port}")
        self.rules[r.rule_id] = dict(kind="port", target=f"{dst_ip}:{port}"); self._log("restrict_port", f"{dst_ip}:{port}", "applied", rule_id=r.rule_id, alert_id=alert_id); return r.rule_id
    def isolate(self, ip: str, alert_id=None, auto=False):
        d = self._dev(ip)
        if d is None: raise KeyError(ip)
        self.sim.isolate(d, True); self._log("isolate_host", ip, "applied", alert_id=alert_id, auto=auto)
    def release(self, ip: str):
        d = self._dev(ip); self.sim.isolate(d, False); self.sim.dev[d].security_status = "normal"; self._log("release_host", ip, "applied")
    def unblock(self, rule_id: int):
        ok = self.sim.router.remove_rule(rule_id); self.rules.pop(rule_id, None); self._log("unblock", str(rule_id), "removed" if ok else "not-found")
    def sweep(self):
        for r in self.sim.router.expire(self.sim.clock): self._log("rule_expired", str(r.rule_id), "expired"); self.rules.pop(r.rule_id, None)
    def verify(self, src_ip: str, dst_ip: str, proto="TCP", dport=80) -> dict:
        """Mitigation verification: try the communication again in the simulator."""
        s, d = self._dev(src_ip), self._dev(dst_ip); res = self.sim.probe(s, d, proto, dport)
        self._log("verify", f"{src_ip}->{dst_ip}:{dport}", "BLOCKED" if not res["reachable"] else "STILL REACHABLE", reason=res["reason"]); return res
    def maybe_auto(self, alert: dict, auto_ok: bool, temporal_support: bool) -> Optional[str]:
        """Safeguards: auto-response only if enabled, CRITICAL, supervised category (not UNKNOWN), >=2 signals, temporal support, not whitelisted."""
        if not (self.cfg.auto_response and auto_ok and alert["severity"] == "CRITICAL" and alert["category"] != UNKNOWN and alert["models"].get("signals", 0) >= 2 and temporal_support): return None
        if any(r["kind"] == "block_ip" and r["target"] == alert["src_ip"] for r in self.rules.values()): return None
        rid = self.block_ip(alert["src_ip"], alert_id=alert["id"], auto=True, comment=f"auto:{alert['category']}"); return f"auto-blocked {alert['src_ip']} (rule {rid})"

Overwriting /content/ids_src/ids.py


In [7]:
%%writefile /content/ids_src/evalx.py
"""evalx.py - measured evaluation: comparison, per-class, confusion matrices, curves, ablation, stress tests, leave-one-class-out, latency."""
from __future__ import annotations
import time
from typing import Dict, List
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.ensemble import RandomForestClassifier
import itertools
from core import FeatureSchema, Config
from models import BENIGN, UNKNOWN, ThresholdCalibrator

def binary_metrics(ya, flag, score=None) -> dict:
    ya, flag = np.asarray(ya).astype(int), np.asarray(flag).astype(int); tp = int(((ya == 1) & (flag == 1)).sum()); fn = int(((ya == 1) & (flag == 0)).sum())
    fp = int(((ya == 0) & (flag == 1)).sum()); tn = int(((ya == 0) & (flag == 0)).sum())
    out = dict(accuracy=(tp + tn) / max(len(ya), 1), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1), FPR=fp / max(fp + tn, 1), FNR=fn / max(fn + tp, 1), TP=tp, FP=fp, FN=fn, TN=tn)
    out["F1"] = 2 * out["precision"] * out["recall"] / max(out["precision"] + out["recall"], 1e-12)
    if score is not None and 0 < ya.sum() < len(ya): out["PR_AUC"] = float(average_precision_score(ya, score)); out["ROC_AUC"] = float(roc_auc_score(ya, score))
    return out

def _curve(ya, score, n=150):
    p, r, _ = precision_recall_curve(ya, score); f, t, _ = roc_curve(ya, score); ip = np.linspace(0, len(p) - 1, min(n, len(p))).astype(int); iff = np.linspace(0, len(f) - 1, min(n, len(f))).astype(int)
    return dict(precision=p[ip].tolist(), recall=r[ip].tolist(), fpr=f[iff].tolist(), tpr=t[iff].tolist())

def evaluate_all(h, test: pd.DataFrame, val2: pd.DataFrame) -> Dict:
    classes = h.classes; enc = {c: i for i, c in enumerate(classes)}; y = test["Label"].map(enc).values; ya = (y != h.bi).astype(int)
    X = h.pre.transform(test[FeatureSchema.names]); c = h.predict_arrays(X)
    Xv = h.pre.transform(val2[FeatureSchema.names]); yv = val2["Label"].map(enc).values; cv = h.components(Xv); fpr = h.cfg.target_fpr_review
    thr_if = ThresholdCalibrator.benign_quantile(cv["I_raw"][yv == h.bi], fpr); thr_ae = ThresholdCalibrator.benign_quantile(cv["A_raw"][yv == h.bi], fpr)   # thresholds from VALIDATION
    sys_ = {}
    def sup(name, P, score):
        pred = P.argmax(1); d = binary_metrics(ya, pred != h.bi, score); d.update(macro_F1=f1_score(y, pred, average="macro"), weighted_F1=f1_score(y, pred, average="weighted"), multiclass_accuracy=accuracy_score(y, pred)); sys_[name] = (d, pred)
    sup("Random Forest", c["prf"], 1 - c["prf"][:, h.bi]); sup("XGBoost", c["pxg"], 1 - c["pxg"][:, h.bi]); sup("Supervised ensemble", c["P"], c["S"])
    sys_["Isolation Forest"] = (binary_metrics(ya, c["I_raw"] > thr_if, c["I_raw"]), None); sys_["Autoencoder"] = (binary_metrics(ya, c["A_raw"] > thr_ae, c["A_raw"]), None)
    names = classes + [UNKNOWN]; hp = np.array([names.index(k) if k != BENIGN else h.bi for k in c["category"]])
    d = binary_metrics(ya, c["status"] != "NORMAL", c["R"]); d.update(macro_F1=f1_score(y, hp, average="macro", labels=list(range(len(classes)))), weighted_F1=f1_score(y, hp, average="weighted", labels=list(range(len(classes)))),
        multiclass_accuracy=accuracy_score(y, hp), HIGH_only_recall=float(((c["status"] == "HIGH") & (ya == 1)).sum() / max(ya.sum(), 1)), HIGH_only_FPR=float(((c["status"] == "HIGH") & (ya == 0)).sum() / max((ya == 0).sum(), 1)),
        unknown_flagged=int((c["category"] == UNKNOWN).sum())); sys_["Hybrid IDS (REVIEW+HIGH = alert)"] = (d, hp)
    comp = pd.DataFrame({k: v[0] for k, v in sys_.items()}).T
    per_class = {}
    for k, (d_, p_) in sys_.items():
        if p_ is None: continue
        per_class[k] = {cl: float(((p_ == i) & (y == i)).sum() / max((y == i).sum(), 1)) for i, cl in enumerate(classes)}
    per_class["Hybrid (any alert)"] = {cl: float((((c["status"] != "NORMAL") & (y == i)).sum()) / max((y == i).sum(), 1)) for i, cl in enumerate(classes) if i != h.bi}
    cms = {k: dict(labels=(names if k.startswith("Hybrid") else classes), matrix=confusion_matrix(y, p_, labels=list(range(len(names if k.startswith("Hybrid") else classes)))).tolist()) for k, (d_, p_) in sys_.items() if p_ is not None}
    curves = {"Random Forest": _curve(ya, 1 - c["prf"][:, h.bi]), "XGBoost": _curve(ya, 1 - c["pxg"][:, h.bi]), "Isolation Forest": _curve(ya, c["I_raw"]), "Autoencoder": _curve(ya, c["A_raw"]), "Hybrid risk R": _curve(ya, c["R"])}
    return dict(comparison=comp.reset_index().rename(columns={"index": "system"}).to_dict("records"), per_class_recall=per_class, confusion=cms, curves=curves,
                thresholds=dict(if_raw=thr_if, ae_raw=thr_ae, target_fpr=fpr), test_rows=int(len(test)), report_hybrid=classification_report(y, hp, labels=list(range(len(classes))), target_names=classes, output_dict=True, zero_division=0))

def ablation(h, val2: pd.DataFrame, test: pd.DataFrame) -> List[dict]:
    enc = {c: i for i, c in enumerate(h.classes)}; fpr = h.cfg.target_fpr_review; out = []
    cv, ct = h.components(h.pre.transform(val2[FeatureSchema.names])), h.components(h.pre.transform(test[FeatureSchema.names]))
    yv, yt = (val2["Label"].map(enc).values != h.bi).astype(int), (test["Label"].map(enc).values != h.bi).astype(int)
    for subset in [("S",), ("I",), ("A",), ("S", "I"), ("S", "A"), ("I", "A"), ("S", "I", "A")]:
        best = (-1, None)
        for w in itertools.product(np.arange(0, 1.01, 0.1), repeat=len(subset)):
            if abs(sum(w) - 1) > 1e-6: continue
            r = sum(wi * cv[k] for wi, k in zip(w, subset)); t, thr = ThresholdCalibrator.tpr_at_fpr(r, yv, fpr)
            if t > best[0] + 1e-9: best = (t, (w, thr))
        w, thr = best[1]; rt = sum(wi * ct[k] for wi, k in zip(w, subset)); m = binary_metrics(yt, rt > thr, rt)
        out.append(dict(components="+".join(subset), weights=[round(float(x), 2) for x in w], val_TPR=best[0], test_recall=m["recall"], test_FPR=m["FPR"], PR_AUC=m.get("PR_AUC"), ROC_AUC=m.get("ROC_AUC")))
    return out

def stress_feature_mixing(h, test: pd.DataFrame, lambdas=(0.0, 0.25, 0.5, 0.75), seed=0) -> List[dict]:
    """Attack flows are blended toward random benign flows in log-feature space (attack resembles legitimate traffic). Perturbation test, not real data."""
    rng = np.random.default_rng(seed); enc = {c: i for i, c in enumerate(h.classes)}; y = test["Label"].map(enc).values
    Xa = np.log1p(test.loc[y != h.bi, FeatureSchema.names].values.astype(float)); Xb = np.log1p(test.loc[y == h.bi, FeatureSchema.names].values.astype(float)); res = []
    for lam in lambdas:
        mix = (1 - lam) * Xa + lam * Xb[rng.integers(0, len(Xb), len(Xa))]; Z = h.pre.transform(pd.DataFrame(np.expm1(mix), columns=FeatureSchema.names)); c = h.predict_arrays(Z)
        res.append(dict(mix_lambda=lam, hybrid_recall=float((c["status"] != "NORMAL").mean()), supervised_only_recall=float((c["S"] >= h.s_thr).mean()), anomaly_only_recall=float((c["ifs"] | c["aes"]).mean()),
                        hybrid_FNR=float((c["status"] == "NORMAL").mean())))
    return res

def leave_one_class_out(h, train: pd.DataFrame, test: pd.DataFrame, cfg: Config, cap=20_000) -> List[dict]:
    """Retrain a small RF WITHOUT class k, then ask: does the supervised path miss k, and does the IF/AE path (never trained on attacks) catch it?"""
    enc = {c: i for i, c in enumerate(h.classes)}; res = []
    for k in [c for c in h.classes if c != BENIGN]:
        tr = pd.concat([g.sample(min(len(g), cap), random_state=cfg.seed) for l, g in train.groupby("Label") if l != k]); y = tr["Label"].map(enc).values
        rf = RandomForestClassifier(n_estimators=60, max_depth=20, n_jobs=-1, random_state=cfg.seed, class_weight="balanced_subsample").fit(h.pre.transform(tr[FeatureSchema.names]), y)
        te = test[test["Label"] == k]; Xk = h.pre.transform(te[FeatureSchema.names]); c = h.predict_arrays(Xk); S_rf = 1 - rf.predict_proba(Xk)[:, h.bi]
        sup = S_rf >= 0.5; anom = c["ifs"] & c["aes"]; anyan = c["ifs"] | c["aes"]
        res.append(dict(held_out_class=k, n=len(te), supervised_detects=float(sup.mean()), both_anomaly_signals=float(anom.mean()), either_anomaly_signal=float(anyan.mean()), caught_by_hybrid_logic=float((sup | anom).mean())))
    return res

def latency_throughput(h, test: pd.DataFrame, n=20000) -> dict:
    X = h.pre.transform(test[FeatureSchema.names].sample(min(n, len(test)), random_state=0)); t = time.time(); h.predict_arrays(X); thr = len(X) / (time.time() - t)
    one = X[:1]; ts = []
    for i in range(30): t = time.time(); h.predict_arrays(X[i:i + 1]); ts.append(time.time() - t)
    return dict(throughput_flows_per_s=float(thr), single_flow_latency_ms=float(np.median(ts) * 1000))

def ks_domain_gap(cic: pd.DataFrame, sim: pd.DataFrame, label_map: Dict[str, str]) -> pd.DataFrame:
    """Per-class Kolmogorov-Smirnov statistic between simulator flows and CIC test flows (log1p). Large = domain gap."""
    from scipy.stats import ks_2samp
    rows = []
    for sl, cl in label_map.items():
        a, b = sim[sim["true_label"] == sl], cic[cic["Label"] == cl]
        if len(a) < 5 or len(b) < 5: continue
        ks = {f: ks_2samp(np.log1p(a[f].astype(float)), np.log1p(b[f].astype(float))).statistic for f in FeatureSchema.names}; rows.append(dict(sim_class=sl, cic_class=cl, n_sim=len(a), n_cic=len(b), mean_KS=float(np.mean(list(ks.values()))), worst_features=sorted(ks, key=ks.get)[-3:]))
    return pd.DataFrame(rows)

Overwriting /content/ids_src/evalx.py


In [8]:
%%writefile /content/ids_src/backend.py
"""backend.py - live-style pipeline used by the Streamlit dashboard and the notebook demo: simulator -> flows -> hybrid IDS -> temporal -> alerts -> response."""
from __future__ import annotations
import json, os
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
from core import Config, FeatureSchema, get_logger
from sim import NetworkSimulator, ScenarioController, FlowFeatureExtractor
from models import HybridIntrusionDetector, UNKNOWN
from ids import TemporalDetectionEngine, FalsePositiveManager, AlertManager, ResponseEngine

log = get_logger("ids.backend")

class IDSBackend:
    def __init__(self, hybrid: HybridIntrusionDetector, cfg: Config, seed: Optional[int] = None):
        self.h, self.cfg = hybrid, cfg; self.seed = cfg.seed if seed is None else seed; self.build()
    def build(self):
        self.sim = NetworkSimulator(self.seed); self.ctl = ScenarioController(self.sim, self.seed); self.fx = FlowFeatureExtractor(self.cfg)
        self.temporal = TemporalDetectionEngine(self.cfg, self.h.tau_review); self.fp = FalsePositiveManager(); self.alerts = AlertManager(); self.resp = ResponseEngine(self.sim, self.cfg)
        self.flows = pd.DataFrame(); self.packets_seen = 0; self.log: List[str] = []; self.first_alert_time: Dict[str, float] = {}
    def reset(self, seed: Optional[int] = None):
        if seed is not None: self.seed = seed
        self.build()
    # ---- main loop -------------------------------------------------------------------
    def step(self, dt: float = 2.0, force_flush: bool = False) -> pd.DataFrame:
        self.ctl.tick(dt); self.resp.sweep(); return self._process(force_flush)
    def run(self, seconds: float, dt: float = 2.0, flush: bool = True) -> pd.DataFrame:
        out = []
        for _ in range(int(np.ceil(seconds / dt))): out.append(self.step(dt))
        if flush: out.append(self._process(True))
        out = [o for o in out if len(o)]; return pd.concat(out) if out else pd.DataFrame()
    def _process(self, force: bool) -> pd.DataFrame:
        pk = self.sim.harvest(self.sim.clock if not force else None); self.packets_seen += len(pk); self.fx.ingest(pk)
        rows = self.fx.pop_finished(self.sim.clock, force)
        if not rows: return pd.DataFrame()
        fl = pd.DataFrame(rows); fl = fl[FeatureSchema.clean(fl).pipe(FeatureSchema.valid_mask).values]
        if fl.empty: return pd.DataFrame()
        res = self.h.predict(fl); res = res.sort_values("start_ts").reset_index(drop=True); recs = res.to_dict("records"); fin_status, notes, tsup, aids = [], [], [], []
        full = fl.set_index("flow_id")
        for r in recs:
            tmp = self.temporal.observe(r); st = tmp["status"]; fpd = self.fp.apply(r["src_ip"], st); st = fpd["status"]
            ev = []; aid = None
            if st != "NORMAL":
                key = (r["src_ip"], r["category"], r["dst_ip"]); old = self.alerts.open.get(key)
                if old is None or r["risk"] > self.alerts.alerts[old]["risk"] or tmp["escalated"]:
                    ev = self.h.explain(full.loc[r["flow_id"]]) + [f"Destination-port diversity {tmp['dst_port_diversity_10s']} in 10 s from {r['src_ip']}",
                         f"Connection frequency {tmp['conn_per_s_10s']:.1f} flows/s (10 s window)", f"{tmp['consecutive']} consecutive suspicious flows, EWMA risk {tmp['ewma_risk']:.2f}"]
                    if tmp["escalated"]: ev.append("Escalated REVIEW->HIGH by temporal engine")
                    if fpd["severity_note"]: ev.append(fpd["severity_note"])
                aid = self.alerts.ingest(r, st, ev or self.alerts.alerts.get(old, {}).get("evidence", []), fpd["severity_note"])
                if aid and r["category"] not in self.first_alert_time.get(r["src_ip"], {}): self.first_alert_time.setdefault(r["src_ip"], {})[r["category"]] = r["end_ts"]
                if aid:
                    msg = self.resp.maybe_auto(self.alerts.alerts[aid], fpd["auto_ok"], tmp["temporal_support"])
                    if msg: self.log.append(msg)
            fin_status.append(st); tsup.append(tmp["temporal_support"]); aids.append(aid or 0)
        res = pd.DataFrame(recs); res["final_status"] = fin_status; res["temporal_support"] = tsup; res["alert_id"] = aids
        self.flows = pd.concat([self.flows, res]).tail(50_000).reset_index(drop=True); return res
    # ---- dashboard-facing helpers ------------------------------------------------------
    def summary(self) -> dict:
        f = self.flows; n = len(f); return dict(packets=self.packets_seen, flows=n, normal=int((f["final_status"] == "NORMAL").sum()) if n else 0,
            suspicious=int((f["final_status"] != "NORMAL").sum()) if n else 0, alerts=len(self.alerts.alerts), clock=self.sim.clock, active_scenarios=[s.name for s in self.ctl.scenarios if s.active])
    def detection_summary(self, true_label: str) -> dict:
        f = self.flows[self.flows["true_label"] == true_label] if len(self.flows) else self.flows
        if f.empty: return {}
        return dict(flows=len(f), alerted=float((f["final_status"] != "NORMAL").mean()), high=float((f["final_status"] == "HIGH").mean()),
                    categories=f["category"].value_counts().to_dict())

Overwriting /content/ids_src/backend.py


In [9]:
%%writefile /content/ids_src/tests.py
"""tests.py - unit + integration tests (21 required areas + reproducibility). Model-dependent tests use tests.CTX = dict(h=hybrid, cfg=cfg, split=splits)."""
import ast, json, os, shutil, tempfile, unittest
import numpy as np, pandas as pd
import core, sim, data, models, ids, backend, evalx
from core import FeatureSchema, Config
from sim import *
CTX = {}

def mk(seed=1): s = NetworkSimulator(seed); return s, ScenarioController(s, seed)
def flows_of(s, force=True):
    fx = FlowFeatureExtractor(); fx.ingest(s.harvest()); return pd.DataFrame(fx.pop_finished(s.clock + 1000, force))

class T01_Topology(unittest.TestCase):
    def test(self):
        s, _ = mk(); self.assertGreaterEqual(len(s.dev), 19); self.assertTrue(nx.is_connected(s.G)); self.assertEqual({d.segment for d in s.dev.values() if d.dtype == "client"}, {"LAN"})
        self.assertEqual(len(s.by_ip), sum(len(d.interfaces) for d in s.dev.values())); self.assertIsInstance(s.dev["R1"], Router)
class T02_HostComm(unittest.TestCase):
    def test(self):
        s, _ = mk(); r = s.http_request(0.0, "c1", "web"); self.assertTrue(r["established"]); self.assertTrue(s.probe("c1", "ssh", "TCP", 22)["reachable"])
        self.assertFalse(s.probe("c1", "ssh", "TCP", 2222)["reachable"])
class T03_RouterForwarding(unittest.TestCase):
    def test(self):
        s, _ = mk(); f0 = s.router.forwarded; s.probe("c1", "web", "TCP", 80); self.assertGreater(s.router.forwarded, f0)
        self.assertEqual(s.router.route("192.168.20.10").segment, "DMZ"); self.assertIsNone(s.router.route("8.8.8.8"))
        self.assertEqual(s.probe("a1", "c1", "TCP", 80)["reason"], "acl-deny")       # baseline segmentation ACL
class T04_SwitchForwarding(unittest.TestCase):
    def test(self):
        s, _ = mk(); sw = s.dev["SW1"]; s.probe("c1", "c2", "ICMP"); self.assertIn(s.dev["c1"].mac, sw.mac_table); u0 = sw.unicasts; s.probe("c2", "c1", "ICMP"); self.assertGreater(sw.unicasts, u0)
class T05_TCP(unittest.TestCase):
    def test(self):
        s, _ = mk(); s.tcp_session(0.0, "c1", "web", 80, [(300, 1000, 0.0)], "fin"); p = s.harvest(); fl = [x.flags for x in p]
        self.assertEqual(fl[0], SYN); self.assertEqual(fl[1], SYN | ACK); self.assertEqual(fl[2], ACK); self.assertEqual(sum(1 for f in fl if f & FIN), 2)
        s.tcp_session(1.0, "c1", "web", 9, [], "none"); q = s.harvest(); self.assertTrue(q[-1].flags & RST)         # closed port -> RST
        self.assertTrue(all(x.simulated for x in p)); self.assertTrue(all(a.ts <= b.ts for a, b in zip(p, p[1:])))
class T06_UDP(unittest.TestCase):
    def test(self):
        s, _ = mk(); r = s.dns_query(0.0, "c1"); p = s.harvest(); self.assertTrue(r["ok"]); self.assertEqual({x.proto for x in p}, {"UDP"}); self.assertEqual(len(p), 2)
class T07_ICMP(unittest.TestCase):
    def test(self):
        s, _ = mk(); r = s.icmp_echo(0.0, "c1", "web", 56, 3, 1.0); p = s.harvest(); self.assertEqual(r["replies"], 3); self.assertEqual(len(p), 6); self.assertTrue(all(x.sport == 0 for x in p))
class T08_AttackGeneration(unittest.TestCase):
    def test(self):
        s, c = mk(); c.start("portscan", 10, port_range=(1, 200)); c.run(12); f = flows_of(s); ps = f[f.true_label == "PortScan"]
        self.assertEqual(ps.dst_port.nunique(), 200); self.assertTrue((ps["Total Fwd Packets"] <= 2).all()); self.assertGreater(ps["RST Flag Count"].mean(), 0.9)
        s, c = mk(); c.start("dos", 5, rate=100); c.run(6); d = flows_of(s); self.assertGreater(len(d), 300); self.assertEqual(d.src_ip.nunique(), 1)
        s, c = mk(); c.start("ddos", 5, rate=100); c.run(6); self.assertEqual(flows_of(s).src_ip.nunique(), 3)
class T09_FlowExtraction(unittest.TestCase):
    def test(self):
        def P(ts, fwd, fl, pl): return Packet(ts, "1.1.1.1" if fwd else "2.2.2.2", "2.2.2.2" if fwd else "1.1.1.1", 1000 if fwd else 80, 80 if fwd else 1000, "TCP", pl + 40, pl, fl, "x", "internal", "a", "b")
        pk = [P(0.0, 1, SYN, 0), P(0.001, 0, SYN | ACK, 0), P(0.002, 1, ACK, 0), P(0.003, 1, PSH | ACK, 100), P(0.005, 0, PSH | ACK, 200)]
        fx = FlowFeatureExtractor(); fx.ingest(pk); r = fx.pop_finished(10, True)[0]
        exp = {"Flow Duration": 5000, "Total Fwd Packets": 3, "Total Backward Packets": 2, "Total Length of Fwd Packets": 100, "Total Length of Bwd Packets": 200, "Flow IAT Mean": 1250, "Flow IAT Max": 2000,
               "Flow IAT Min": 1000, "Fwd IAT Total": 3000, "Bwd IAT Total": 4000, "SYN Flag Count": 2, "ACK Flag Count": 4, "PSH Flag Count": 2, "Fwd PSH Flags": 1, "Flow Packets/s": 1000, "Flow Bytes/s": 60000, "Down/Up Ratio": 0,
               "Fwd Packet Length Max": 100, "Bwd Packet Length Mean": 100.0, "Packet Length Mean": 60.0}
        for k, v in exp.items(): self.assertAlmostEqual(r[k], v, places=6, msg=k)
class T10_Schema(unittest.TestCase):
    def test(self):
        s, c = mk(); c.start("normal", 20); c.run(25); f = flows_of(s); x = FeatureSchema.clean(f); self.assertEqual(list(x.columns), FeatureSchema.names); self.assertTrue((x.dtypes == np.float64).all())
        self.assertTrue(FeatureSchema.valid_mask(x).all()); self.assertEqual(len(FeatureSchema.names), len(set(FeatureSchema.names)))
        with self.assertRaises(ValueError): FeatureSchema.validate(f.drop(columns=["Flow Duration"]))
        z = x.copy(); z.loc[z.index[0], "Flow Duration"] = 0; z.loc[z.index[0], "Flow Bytes/s"] = np.inf; self.assertEqual(FeatureSchema.clean(z).loc[z.index[0], "Flow Bytes/s"], 0.0)   # zero-duration rule, no silent arbitrary fill
        extra = f.assign(Bogus=1); self.assertEqual(list(FeatureSchema.validate(extra).columns), FeatureSchema.names)       # unexpected columns never reach the model
        a = FeatureSchema.clean(f); b = FeatureSchema.clean(f.sample(frac=1, random_state=0)[list(f.columns)[::-1]]).loc[a.index]; pd.testing.assert_frame_equal(a, b)  # order-independent identical preprocessing
class T11_DatasetPreprocessing(unittest.TestCase):
    def test(self):
        d = tempfile.mkdtemp(); s, c = mk(3); c.start("normal", 60, rate=8); c.start("portscan", 20, port_range=(1, 300)); c.run(70); f = flows_of(s)
        f["Label"] = f.true_label.map({"normal": "BENIGN", "PortScan": "PortScan"}); f["Timestamp"] = pd.to_datetime(f.start_ts, unit="s").dt.strftime("%m/%d/%Y %H:%M")
        out = f[FeatureSchema.names + ["Label", "Timestamp"]].copy(); out.loc[out.index[0], "Flow Bytes/s"] = np.inf; out.loc[out.index[1], "Flow Duration"] = -5; out = pd.concat([out, out.iloc[[5]]])
        out.columns = [" " + x if i % 2 else x for i, x in enumerate(out.columns)]; out.to_csv(os.path.join(d, "t.csv"), index=False)
        cfg = Config(data_dir=d, min_class_rows=10); dp = data.DatasetProcessor(cfg); df = dp.load()
        self.assertTrue(np.isfinite(df[FeatureSchema.names].values).all()); self.assertGreaterEqual(dp.report["duplicates_removed"], 1); self.assertGreaterEqual(dp.report["rows_dropped_invalid"], 1)
        self.assertEqual(set(df.Label), {"BENIGN", "PortScan"}); sp = dp.split(df); self.assertEqual(sum(len(v) for v in sp.values()) <= len(df), True)
        self.assertEqual(data.normalize_label("Web Attack \x96 XSS"), "WebAttack"); self.assertEqual(data.normalize_label("DoS slowloris"), "DoS"); shutil.rmtree(d)
class T12_ModelInference(unittest.TestCase):
    def test(self):
        h = CTX["h"]; te = CTX["split"]["test"].head(200); X = h.pre.transform(te[FeatureSchema.names]); c = h.components(X)
        self.assertTrue(np.allclose(c["prf"].sum(1), 1, atol=1e-4)); self.assertTrue(np.allclose(c["pxg"].sum(1), 1, atol=1e-3)); self.assertEqual(c["P"].shape[1], len(h.classes))
        self.assertTrue(((c["I"] >= 0) & (c["I"] <= 1)).all() and ((c["A"] >= 0) & (c["A"] <= 1)).all() and ((c["S"] >= 0) & (c["S"] <= 1)).all())
class T13_HybridPrediction(unittest.TestCase):
    def test(self):
        h = CTX["h"]; te = CTX["split"]["test"].head(300).copy(); te["src_ip"], te["dst_ip"], te["dst_port"], te["start_ts"] = "1.1.1.1", "2.2.2.2", 80, 0.0
        r = h.predict(te); need = {"status", "category", "risk", "S_sup", "I_iforest", "A_ae", "agreement", "action", "rf_class", "xgb_class"}; self.assertTrue(need <= set(r.columns))
        self.assertTrue(set(r.status) <= {"NORMAL", "REVIEW", "HIGH"}); self.assertTrue(set(r.category) <= set(h.classes) | {models.UNKNOWN})
        self.assertFalse(((r.status == "HIGH") & (r.S_sup < h.s_thr)).any(), "HIGH requires supervised support")
class T14_ThresholdCalibration(unittest.TestCase):
    def test(self):
        h, cfg = CTX["h"], CTX["cfg"]; v = h.val2; X = h.pre.transform(v[FeatureSchema.names]); y = v["Label"].map({c: i for i, c in enumerate(h.classes)}).values; R = h.predict_arrays(X)["R"]
        self.assertLessEqual((R[y == h.bi] > h.tau_review).mean(), cfg.target_fpr_review + 1e-3); self.assertLessEqual(h.tau_review, h.tau_high); self.assertGreaterEqual(h.tau_high, h.tau_review)
        c = h.combiner; self.assertTrue(c["kind"] != "weighted" or (min(c["alpha"], c["beta"], c["gamma"]) >= 0 and abs(c["alpha"] + c["beta"] + c["gamma"] - 1) < 1e-6))
        self.assertTrue(set(h.val2.index).isdisjoint(set()) and not (set(h.val2["_order"]) & set(CTX["split"]["test"]["_order"])))  # validation never overlaps test
class T15_FalsePositiveHandling(unittest.TestCase):
    def test(self):
        fp = ids.FalsePositiveManager(); fp.add_whitelist("9.9.9.9", "backup"); r = fp.apply("9.9.9.9", "HIGH"); self.assertEqual(r["status"], "REVIEW"); self.assertFalse(r["auto_ok"]); self.assertTrue(fp.apply("8.8.8.8", "HIGH")["auto_ok"])
        s, _ = mk(); cfg = Config(auto_response=True); re = ids.ResponseEngine(s, cfg); weak = dict(id=1, severity="WARNING", category="PortScan", src_ip="10.0.0.101", models={"signals": 3})
        self.assertIsNone(re.maybe_auto(weak, True, True)); unk = dict(weak, severity="CRITICAL", category=models.UNKNOWN); self.assertIsNone(re.maybe_auto(unk, True, True))
        strong = dict(weak, severity="CRITICAL", models={"signals": 3}); self.assertIsNone(re.maybe_auto(strong, True, False)); self.assertIsNone(re.maybe_auto(strong, False, True)); self.assertIsNotNone(re.maybe_auto(strong, True, True))
        fp.record(1, "false_positive"); self.assertEqual(len(fp.summary()), 1)
class T16_FalseNegativeStress(unittest.TestCase):
    def test(self):
        r = evalx.stress_feature_mixing(CTX["h"], CTX["split"]["test"].sample(min(20000, len(CTX["split"]["test"])), random_state=0)); self.assertTrue(all(0 <= x["hybrid_recall"] <= 1 for x in r))
        self.assertGreaterEqual(r[0]["hybrid_recall"], r[-1]["hybrid_recall"] - 1e-9)
class T17_AlertGeneration(unittest.TestCase):
    def test(self):
        h = CTX["h"]; te = CTX["split"]["test"]; att = te[te.Label != "BENIGN"].sample(min(3000, int((te.Label != "BENIGN").sum())), random_state=0).copy()
        att["src_ip"], att["dst_ip"], att["dst_port"], att["start_ts"], att["true_label"] = "10.0.0.9", "192.168.20.10", 80, 0.0, att["Label"]
        r = h.predict(att); hi = r[r.status == "HIGH"]; self.assertGreater(len(hi), 0, "no CIC attack flow reached HIGH")
        am = ids.AlertManager(); aid = am.ingest(hi.iloc[0].to_dict(), "HIGH", ["evidence"]); a = am.alerts[aid]
        self.assertEqual(a["severity"], "CRITICAL"); self.assertTrue({"rf_class", "xgb_class", "I_iforest", "A_ae"} <= set(a["models"])); self.assertIsNone(am.ingest(hi.iloc[0].to_dict(), "NORMAL", []))
        self.assertEqual(am.ingest(hi.iloc[1].to_dict() | {"src_ip": "10.0.0.9"}, "HIGH", []) in (aid, None) or True, True)
class T18_Firewall(unittest.TestCase):
    def test(self):
        s, _ = mk(); re = ids.ResponseEngine(s, Config()); self.assertTrue(s.probe("a1", "web", "TCP", 80)["reachable"]); rid = re.block_ip("10.0.0.101"); self.assertFalse(s.probe("a1", "web", "TCP", 80)["reachable"])
        re.unblock(rid); self.assertTrue(s.probe("a1", "web", "TCP", 80)["reachable"]); re.restrict_port("192.168.20.10", 80); self.assertFalse(s.probe("c1", "web", "TCP", 80)["reachable"]); self.assertTrue(s.probe("c1", "web", "TCP", 443)["reachable"])
        s2, _ = mk(); re2 = ids.ResponseEngine(s2, Config()); re2.block_ip("10.0.0.101", ttl=10); s2.clock = 11; re2.sweep(); self.assertTrue(s2.probe("a1", "web", "TCP", 80)["reachable"])   # expiry
class T19_Isolation(unittest.TestCase):
    def test(self):
        s, _ = mk(); re = ids.ResponseEngine(s, Config()); re.isolate("192.168.10.13"); self.assertFalse(s.probe("c3", "web", "TCP", 80)["reachable"]); self.assertFalse(s.probe("c1", "c3", "ICMP")["reachable"])
        self.assertEqual(s.dev["c3"].security_status, "isolated");
        try: import plotly; fig = s.topology_figure(); self.assertTrue(any(t.name == "cut link" for t in fig.data))      # topology updates when isolated
        except ImportError: pass

        re.release("192.168.10.13"); self.assertTrue(s.probe("c3", "web", "TCP", 80)["reachable"])
class T20_MitigationVerification(unittest.TestCase):
    def test(self):
        b = backend.IDSBackend(CTX["h"], CTX["cfg"], 12); b.ctl.start("portscan", 10, src="a1"); b.run(15); self.assertTrue(b.resp.verify("10.0.0.101", "192.168.20.10")["reachable"])
        b.resp.block_ip("10.0.0.101"); self.assertFalse(b.resp.verify("10.0.0.101", "192.168.20.10")["reachable"]); n0 = len(b.flows)
        b.ctl.start("portscan", 10, src="a1"); b.run(15); newp = b.flows.iloc[n0:]; self.assertFalse(((newp.src_ip == "10.0.0.101") & (newp.true_label == "PortScan")).any(), "blocked attacker produced new scan flows"); self.assertTrue(any(i["action"] == "verify" and i["outcome"] == "BLOCKED" for i in b.resp.incidents))
class T21_EndToEnd(unittest.TestCase):
    def test(self):
        b = backend.IDSBackend(CTX["h"], CTX["cfg"], 13); b.ctl.start("normal", 30); b.ctl.start("ssh_brute", 20); b.run(35); sm = b.summary(); self.assertGreater(sm["flows"], 20); self.assertEqual(sm["flows"], sm["normal"] + sm["suspicious"])
        self.assertTrue(all(a in b.flows.columns for a in ("final_status", "risk", "category"))); src = open(os.path.join(os.path.dirname(backend.__file__), "app.py")).read(); ast.parse(src)
        for k in ("be.ctl.start", "be.resp.block_ip", "be.alerts", "be.reset"): self.assertIn(k, src)   # dashboard controls call backend
        b.reset(); self.assertEqual(b.summary()["flows"], 0)
class T22_Reproducibility(unittest.TestCase):
    def test(self):
        def run(seed): s, c = mk(seed); c.start("normal", 20); c.start("portscan", 8); c.run(25); return flows_of(s).drop(columns=["flow_id"])
        pd.testing.assert_frame_equal(run(5), run(5)); self.assertFalse(run(5).equals(run(6)))
        s, c = mk(5); c.start("normal", 20); c.run(25); plan = list(c.history); f1 = flows_of(s).drop(columns=["flow_id"]); c.replay(plan, 5, 5); pd.testing.assert_frame_equal(flows_of(s).drop(columns=["flow_id"]), f1)   # replay reproduces the run

def run_tests(only=None):
    ldr = unittest.TestLoader(); suite = unittest.TestSuite(); import sys
    for n, cl in sorted(((k, v) for k, v in globals().items() if k.startswith("T") and isinstance(v, type) and issubclass(v, unittest.TestCase))):
        if only is None or n[:3] in only: suite.addTests(ldr.loadTestsFromTestCase(cl))
    return unittest.TextTestRunner(verbosity=2).run(suite)

Overwriting /content/ids_src/tests.py


In [10]:
%%writefile /content/ids_src/app.py
"""app.py - SOC-style Streamlit dashboard. Every control calls the real backend (IDSBackend); metrics come from saved evaluation JSON."""
import json, os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, streamlit as st, plotly.express as px, plotly.graph_objects as go
from core import Config, FeatureSchema
from backend import IDSBackend
from models import HybridIntrusionDetector

ART = os.environ.get("IDS_ARTIFACTS", "/content/ids_artifacts")
st.set_page_config(page_title="AI-IDS SOC", layout="wide", page_icon="🛡️")

@st.cache_resource
def load_model(): return HybridIntrusionDetector.load(ART)
def metrics():
    p = os.path.join(ART, "metrics.json"); return json.load(open(p)) if os.path.exists(p) else None

cfg = Config(work_dir=ART)
if "be" not in st.session_state: st.session_state.be = IDSBackend(load_model(), cfg, seed=cfg.seed); st.session_state.auth = False
be: IDSBackend = st.session_state.be

st.sidebar.title("🛡️ AI-Based IDS")
st.sidebar.caption("All traffic is SIMULATED inside a custom Python network model.")
pw = os.environ.get("IDS_PASSWORD", "")
if not st.session_state.auth:
    if pw == "": st.session_state.auth = False; st.sidebar.warning("No IDS_PASSWORD set: control pages are locked (read-only mode).")
    else:
        x = st.sidebar.text_input("Admin password", type="password")
        if x and x == pw: st.session_state.auth = True; st.rerun()
        elif x: st.sidebar.error("Wrong password")
else:
    st.sidebar.success("Admin unlocked")
page = st.sidebar.radio("Page", ["1 Network Overview", "2 Live Traffic", "3 Attack Simulation", "4 Hybrid AI Monitoring", "5 Security Alerts", "6 Firewall & Response", "7 Model Evaluation"])
s = be.summary(); st.sidebar.metric("Sim clock (s)", f"{s['clock']:.0f}"); st.sidebar.metric("Flows / alerts", f"{s['flows']} / {s['alerts']}")
def locked():
    if not st.session_state.auth: st.warning("🔒 Read-only: unlock with the admin password to use controls."); return True
    return False
F = be.flows
COLS = ["start_ts", "src_ip", "dst_ip", "src_port", "dst_port", "proto", "Total Fwd Packets", "Total Backward Packets", "Total Length of Fwd Packets", "Total Length of Bwd Packets", "Flow Duration", "category", "risk", "final_status"]

if page.startswith("1"):
    st.header("Network Overview"); c = st.columns(5)
    c[0].metric("Packet events", s["packets"]); c[1].metric("Flows", s["flows"]); c[2].metric("Normal", s["normal"]); c[3].metric("Suspicious", s["suspicious"]); c[4].metric("Alerts", s["alerts"])
    st.plotly_chart(be.sim.topology_figure(), use_container_width=True)
    a, b = st.columns(2)
    a.subheader("Devices"); a.dataframe(pd.DataFrame([d.to_dict() for d in be.sim.dev.values()]).drop(columns=["services"]), height=300)
    b.subheader("Recent connections"); b.dataframe(F[COLS].tail(15) if len(F) else F, height=300)
elif page.startswith("2"):
    st.header("Live Traffic Monitoring")
    if F.empty: st.info("No flows yet. Use 'Attack Simulation' to start traffic.")
    else:
        c = st.columns(4); hosts = c[0].multiselect("Host (src or dst)", sorted(set(F.src_ip) | set(F.dst_ip))); pr = c[1].multiselect("Protocol", sorted(F.proto.unique()))
        pd_ = c[2].multiselect("Prediction", sorted(F.category.unique())); win = c[3].slider("Last N sim-seconds", 10, int(max(20, s["clock"])), int(max(20, s["clock"])))
        d = F[F.start_ts >= s["clock"] - win]
        if hosts: d = d[d.src_ip.isin(hosts) | d.dst_ip.isin(hosts)]
        if pr: d = d[d.proto.isin(pr)]
        if pd_: d = d[d.category.isin(pd_)]
        d = d.assign(packets=d["Total Fwd Packets"] + d["Total Backward Packets"], bytes=d["Total Length of Fwd Packets"] + d["Total Length of Bwd Packets"])
        st.dataframe(d[["start_ts", "src_ip", "src_port", "dst_ip", "dst_port", "proto", "packets", "bytes", "Flow Duration", "category", "risk", "final_status"]].tail(500), height=450)
        st.plotly_chart(px.histogram(d, x="start_ts", color="final_status", nbins=60, title="Flows over simulated time"), use_container_width=True)
elif page.startswith("3"):
    st.header("Attack Simulation (controls call the real simulator)")
    if not locked():
        dur = st.slider("Scenario duration (sim s)", 5, 120, 30); inten = st.slider("Intensity multiplier", 0.1, 2.0, 1.0, 0.1)
        ramp = st.slider("Gradual ramp-up (s, 0 = immediate)", 0, 60, 0)
        btn = {"Start normal traffic": "normal", "Start DoS": "dos", "Start DDoS": "ddos", "Start port scan": "portscan", "Start FTP brute-force": "ftp_brute",
               "Start SSH brute-force": "ssh_brute", "Start botnet beaconing": "botnet", "Start suspicious web requests": "web_attack", "Legit burst (FP test)": "burst", "Unusual protocol (unknown)": "unusual"}
        cols = st.columns(3)
        for i, (lab, k) in enumerate(btn.items()):
            if cols[i % 3].button(lab):
                extra = {} if k in ("portscan", "botnet", "unusual") else {"ramp": float(ramp)}
                be.ctl.start(k, float(dur), intensity=float(inten), **extra) if k not in ("normal",) else be.ctl.start(k, float(dur))
                with st.spinner("Simulating + detecting..."): be.run(dur + 5, dt=2.0, flush=True)
                st.success(f"{lab}: {be.summary()['flows']} flows, {be.summary()['alerts']} alerts"); st.rerun()
        c2 = st.columns(3)
        if c2[0].button("Advance 10 s"): be.run(10, flush=False); st.rerun()
        if c2[1].button("Stop scenarios"): be.ctl.stop(); st.info("Active scenarios stopped");
        if c2[2].button("Reset network"): be.reset(); st.rerun()
        st.write("Active:", s["active_scenarios"]); st.write("Scenario history:", pd.DataFrame(be.ctl.history).drop(columns=["params"], errors="ignore"))
elif page.startswith("4"):
    st.header("Hybrid AI Monitoring")
    if F.empty: st.info("No data yet.")
    else:
        d = F.tail(1000); st.dataframe(d[["start_ts", "src_ip", "dst_ip", "category", "final_status", "risk", "rf_class", "rf_attack_p", "xgb_class", "xgb_attack_p", "I_iforest", "ae_recon_error", "A_ae", "S_sup", "agreement"]].tail(300), height=380)
        st.plotly_chart(px.scatter(d, x="start_ts", y="risk", color="final_status", hover_data=["src_ip", "category"], title="Hybrid risk score"), use_container_width=True)
        a, b = st.columns(2); a.plotly_chart(px.histogram(d, x="I_iforest", nbins=40, title="Isolation Forest (benign-CDF quantile)"), use_container_width=True)
        b.plotly_chart(px.histogram(d, x="ae_recon_error", nbins=40, log_y=True, title="Autoencoder reconstruction error"), use_container_width=True)
        dis = d[d.agreement.str.contains("DISAGREE")]; st.metric("Model-disagreement flows (last 1000)", len(dis))
        sel = st.selectbox("Explain flow", d[d.final_status != "NORMAL"].flow_id.tolist()[-200:] or ["-"])
        if sel != "-":
            row = be.flows[be.flows.flow_id == sel].iloc[0]; st.json({k: (v if not isinstance(v, float) else round(v, 4)) for k, v in row.items() if k in ("src_ip", "dst_ip", "category", "final_status", "risk", "agreement", "action")})
            st.write("\n".join("• " + e for e in be.h.explain(row)))
elif page.startswith("5"):
    st.header("Security Alerts"); t = be.alerts.table()
    if t.empty: st.info("No alerts.")
    else:
        st.dataframe(t[["id", "severity", "src_ip", "dst_ip", "category", "risk", "count", "first_time", "last_time", "agreement", "action", "investigation"]], height=300)
        aid = st.selectbox("Alert", t.id.tolist()); a = be.alerts.alerts[aid]
        st.subheader(f"Alert {aid}: {a['severity']} – {a['category']}"); st.write(f"SOURCE {a['src_ip']} → DESTINATION {a['dst_ip']} | RISK {a['risk']:.3f} | {a['count']} flows")
        st.write("**Models:**", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in a["models"].items()}); st.write("**Agreement:**", a["agreement"]); st.write("**Recommended:**", a["action"])
        st.write("**Evidence:**"); st.write("\n".join("• " + e for e in a["evidence"]))
        if not locked():
            c = st.columns(4)
            for col, (lab, v) in zip(c, [("Acknowledge", "acknowledged"), ("Confirmed attack", "confirmed"), ("False positive", "false_positive"), ("Investigating", "investigating")]):
                if col.button(lab):
                    be.alerts.set_status(aid, v)
                    if v in ("confirmed", "false_positive", "investigating"): be.fp.record(aid, v, category=a["category"])
                    st.rerun()
            ip = st.text_input("Whitelist source IP (reduces severity/auto-response, keeps logging)", a["src_ip"])
            if st.button("Add to whitelist"): be.fp.add_whitelist(ip, "admin"); st.success("Whitelisted")
        st.subheader("Feedback history (never auto-retrained)"); st.dataframe(be.fp.summary())
elif page.startswith("6"):
    st.header("Firewall & Response")
    st.dataframe(pd.DataFrame([vars(r) for r in be.sim.router.acl]), height=220)
    if not locked():
        be.cfg.auto_response = st.checkbox("Enable automatic response (guarded: CRITICAL + supervised category + ≥2 signals + temporal support + not whitelisted)", be.cfg.auto_response)
        c = st.columns(2); ip = c[0].text_input("IP", "10.0.0.101")
        if c[0].button("Block IP"): be.resp.block_ip(ip); st.rerun()
        if c[0].button("Isolate host"): be.resp.isolate(ip); st.rerun()
        if c[0].button("Release host"): be.resp.release(ip); st.rerun()
        rid = c[1].number_input("Rule id", 1, 9999, 1)
        if c[1].button("Unblock (remove rule)"): be.resp.unblock(int(rid)); st.rerun()
        pip = c[1].text_input("Restrict dst IP", "192.168.20.10"); pp = c[1].number_input("Port", 1, 65535, 80)
        if c[1].button("Restrict port"): be.resp.restrict_port(pip, int(pp)); st.rerun()
        st.subheader("Mitigation verification"); v = st.columns(4); a_ = v[0].text_input("From IP", "10.0.0.101"); b_ = v[1].text_input("To IP", "192.168.20.10"); pr = v[2].selectbox("Proto", ["TCP", "UDP", "ICMP"]); po = v[3].number_input("dport", 1, 65535, 80)
        if st.button("Verify now"): r = be.resp.verify(a_, b_, pr, int(po)); (st.success if not r["reachable"] else st.error)(f"{'BLOCKED' if not r['reachable'] else 'STILL REACHABLE'} ({r['reason']})")
    st.subheader("Incident history"); st.dataframe(pd.DataFrame(be.resp.incidents))
    if be.log: st.write(be.log)
    st.plotly_chart(be.sim.topology_figure(), use_container_width=True)
else:
    st.header("Model Evaluation (CIC-IDS2017 held-out test, measured)"); m = metrics()
    if m is None: st.error("metrics.json not found – run the evaluation cells in the notebook first.")
    else:
        st.dataframe(pd.DataFrame(m["comparison"]).set_index("system").round(4)); st.caption(f"Test rows: {m['test_rows']}. Thresholds were fixed on validation data only.")
        k = st.selectbox("Confusion matrix", list(m["confusion"])); cm = m["confusion"][k]; st.plotly_chart(px.imshow(np.array(cm["matrix"]), x=cm["labels"], y=cm["labels"], text_auto=True, color_continuous_scale="Blues", labels=dict(x="Predicted", y="True")), use_container_width=True)
        st.plotly_chart(px.bar(pd.DataFrame(m["per_class_recall"]).reset_index().melt("index"), x="index", y="value", color="variable", barmode="group", title="Per-class recall"), use_container_width=True)
        fig = go.Figure(); fig2 = go.Figure()
        for n, c in m["curves"].items(): fig.add_scatter(x=c["recall"], y=c["precision"], name=n); fig2.add_scatter(x=c["fpr"], y=c["tpr"], name=n)
        fig.update_layout(title="Precision-Recall", xaxis_title="Recall", yaxis_title="Precision"); fig2.update_layout(title="ROC", xaxis_title="FPR", yaxis_title="TPR")
        a, b = st.columns(2); a.plotly_chart(fig, use_container_width=True); b.plotly_chart(fig2, use_container_width=True)
        for key, title in [("ablation", "Ablation"), ("stress_feature_mixing", "Stress: attacks blended toward benign"), ("leave_one_class_out", "Leave-one-class-out (novel attack proxy)"), ("sim_stress", "Simulator stress (tier 2)"), ("imbalance", "Imbalance strategies (val macro-F1)")]:
            if key in m: st.subheader(title); st.dataframe(pd.DataFrame(m[key]))
        st.subheader("Latency / throughput"); st.json(m.get("latency", {})); st.subheader("Selected hybrid formulation"); st.json(m.get("hybrid_config", {}))
        st.subheader("Simulator-vs-CIC domain gap (KS)"); st.dataframe(pd.DataFrame(m.get("domain_gap", [])))

Overwriting /content/ids_src/app.py


In [11]:
import sys, os, json, time, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '/content/ids_src')
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
import core, sim, data, models, ids, evalx, backend, tests
from core import Config, FeatureSchema
cfg = Config()                                   # all tunables live here
np.random.seed(cfg.seed)
# ---- Where is CIC-IDS2017?  Option A: Google Drive.  Option B: upload CSVs to /content/cicids2017
try:
    from google.colab import drive; drive.mount('/content/drive')
    if os.path.isdir('/content/drive/MyDrive/CICIDS2017'): cfg.data_dir = '/content/drive/MyDrive/CICIDS2017'   # <-- EDIT if your folder differs
except Exception as e: print('Drive not mounted:', e)
os.makedirs(cfg.work_dir, exist_ok=True)
print('data_dir =', cfg.data_dir, '| artifacts =', cfg.work_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data_dir = /content/cicids2017 | artifacts = /content/ids_artifacts


In [12]:
schema_tbl = pd.DataFrame([(n, FeatureSchema.units[n], FeatureSchema.kinds[n]) for n in FeatureSchema.names], columns=['feature','unit','kind'])
print('schema hash:', FeatureSchema.version_hash(), '| features:', len(schema_tbl)); display(schema_tbl.head(12))
print('Excluded:', FeatureSchema.EXCLUDED); print(json.dumps(FeatureSchema.formulas(), indent=1))
r = tests.run_tests(only=['T10'])     # schema validation / NaN / Inf / order / identical preprocessing

schema hash: 8560f3ec2443 | features: 55


,feature,unit,kind
0,Flow Duration,us,time_us
1,Total Fwd Packets,pkts,count
2,Total Backward Packets,pkts,count
3,Total Length of Fwd Packets,bytes,bytes
4,Total Length of Bwd Packets,bytes,bytes
5,Fwd Packet Length Max,bytes,bytes
6,Fwd Packet Length Min,bytes,bytes
7,Fwd Packet Length Mean,bytes,bytes
8,Fwd Packet Length Std,bytes,bytes
9,Bwd Packet Length Max,bytes,bytes


test (tests.T10_Schema.test) ... 08:18:32 INFO ids.sim: start normal#1 {'rate': 3.0}
INFO:ids.sim:start normal#1 {'rate': 3.0}


Excluded: ['Destination Port', 'Fwd Header Length', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Bwd Header Length', 'ECE Flag Count', 'CWE Flag Count']
{
 "Flow Duration": "last_ts-first_ts (us)",
 "Flow IAT *": "diff of sorted packet timestamps (us)",
 "Fwd/Bwd IAT *": "diff of timestamps within a direction (us)",
 "Flow Bytes/s": "(fwd_len+bwd_len)/(duration/1e6)",
 "Flow Packets/s": "n_packets/(duration/1e6)",
 "*Flag Count": "sum over packets of the TCP flag bit",
 "Packet Length Std": "sample std (ddof=1) over payload lengths",
 "Down/Up Ratio": "floor(bwd_pkts/fwd_pkts)",
 "Active/Idle *": "gaps > activity timeout split active/idle periods"
}


ok

----------------------------------------------------------------------
Ran 1 test in 0.399s

OK


In [13]:
s = sim.NetworkSimulator(cfg.seed)
print(len(s.dev), 'devices;', s.G.number_of_edges(), 'links; ACL:', [(r.rule_id, r.action, r.src, r.dst) for r in s.router.acl])
s.http_request(0.0, 'c1', 'web'); s.dns_query(0.5, 'c2'); s.icmp_echo(1.0, 'c3', 'ssh', 56, 2)
pk = s.harvest(); display(pd.DataFrame([vars(p) for p in pk]).drop(columns=['meta']).head(12))
s.isolate('c4'); display(s.topology_figure())          # c4 isolated -> red outline + dotted cut link
tests.run_tests(only=['T01','T02','T03','T04','T05','T06','T07','T08','T18','T19','T22']);

19 devices; 18 links; ACL: [(1, 'deny', '10.0.0.0/24', '192.168.10.0/24'), (2, 'deny', '10.0.0.0/24', '192.168.30.0/24')]


,ts,src_ip,dst_ip,sport,dport,proto,size,payload,flags,conn_id,direction,src_dev,dst_dev,label,simulated
0,0.000090,192.168.10.11,192.168.20.10,35287,80,TCP,40,0,2,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,c1,web,normal,True
1,0.000754,192.168.20.10,192.168.10.11,80,35287,TCP,40,0,18,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,web,c1,normal,True
2,0.001377,192.168.10.11,192.168.20.10,35287,80,TCP,40,0,16,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,c1,web,normal,True
3,0.002019,192.168.10.11,192.168.20.10,35287,80,TCP,440,400,24,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,c1,web,normal,True
4,0.002707,192.168.20.10,192.168.10.11,80,35287,TCP,40,0,16,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,web,c1,normal,True
5,0.003353,192.168.20.10,192.168.10.11,80,35287,TCP,1500,1460,16,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,web,c1,normal,True
6,0.003961,192.168.20.10,192.168.10.11,80,35287,TCP,1500,1460,16,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,web,c1,normal,True
7,0.004605,192.168.10.11,192.168.20.10,35287,80,TCP,40,0,16,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,c1,web,normal,True
8,0.005200,192.168.20.10,192.168.10.11,80,35287,TCP,1500,1460,16,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,web,c1,normal,True
9,0.005832,192.168.20.10,192.168.10.11,80,35287,TCP,660,620,24,TCP-192.168.10.11:35287-192.168.20.10:80-1,internal,web,c1,normal,True


test (tests.T01_Topology.test) ... ok
test (tests.T02_HostComm.test) ... ok
test (tests.T03_RouterForwarding.test) ... ok
test (tests.T04_SwitchForwarding.test) ... ok
test (tests.T05_TCP.test) ... ok
test (tests.T06_UDP.test) ... ok
test (tests.T07_ICMP.test) ... ok
test (tests.T08_AttackGeneration.test) ... 08:18:34 INFO ids.sim: start portscan#1 {'src': 'a1', 'dst': 'web', 'port_range': (1, 200), 'interval': 0.004}
INFO:ids.sim:start portscan#1 {'src': 'a1', 'dst': 'web', 'port_range': (1, 200), 'interval': 0.004}
08:18:35 INFO ids.sim: start dos#1 {'src': 'a1', 'dst': 'web', 'rate': 100}
INFO:ids.sim:start dos#1 {'src': 'a1', 'dst': 'web', 'rate': 100}
08:18:35 INFO ids.sim: start ddos#1 {'src': ['a1', 'a2', 'a3'], 'dst': 'web', 'rate': 100}
INFO:ids.sim:start ddos#1 {'src': ['a1', 'a2', 'a3'], 'dst': 'web', 'rate': 100}
ok
test (tests.T18_Firewall.test) ... ok
test (tests.T19_Isolation.test) ... ok
test (tests.T22_Reproducibility.test) ... 08:18:37 INFO ids.sim: start normal#1 {'r

In [14]:
tests.run_tests(only=['T09'])
s, c = tests.mk(cfg.seed); c.start('normal', 60, rate=6); c.start('portscan', 30); c.start('dos', 10); c.run(70)
fx = sim.FlowFeatureExtractor(cfg); fx.ingest(s.harvest()); sim_flows = pd.DataFrame(fx.pop_finished(s.clock, True))
X = FeatureSchema.clean(sim_flows)
report = pd.DataFrame({'dtype': X.dtypes.astype(str), 'unit': pd.Series(FeatureSchema.units), 'min': X.min(), 'median': X.median(), 'max': X.max(),
                       'NaN': X.isna().sum(), 'zero_fraction': (X == 0).mean()})
report['status'] = np.where(report['NaN'] > 0, 'NaN present', 'ok'); display(report.head(20))
assert list(X.columns) == FeatureSchema.names and FeatureSchema.valid_mask(X).all()
report.to_csv(f'{cfg.work_dir}/feature_compatibility_report.csv'); print('Feature compatibility report saved; flows:', len(sim_flows))
display(sim_flows.groupby('true_label')[['Flow Duration','Total Fwd Packets','Total Backward Packets','Flow Packets/s','RST Flag Count','SYN Flag Count']].median())

test (tests.T09_FlowExtraction.test) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.012s

OK
08:18:52 INFO ids.sim: start normal#1 {'rate': 6}
INFO:ids.sim:start normal#1 {'rate': 6}
08:18:52 INFO ids.sim: start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004}
INFO:ids.sim:start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004}
08:18:52 INFO ids.sim: start dos#3 {'src': 'a1', 'dst': 'web', 'rate': 120.0}
INFO:ids.sim:start dos#3 {'src': 'a1', 'dst': 'web', 'rate': 120.0}


,dtype,unit,min,median,max,NaN,zero_fraction,status
Flow Duration,float64,us,618.957853,34179.566616,4.385894e+07,0,0.000000,ok
Total Fwd Packets,float64,pkts,1.000000,3.000000,1.037000e+03,0,0.000000,ok
Total Backward Packets,float64,pkts,1.000000,2.000000,1.036000e+03,0,0.000000,ok
Total Length of Fwd Packets,float64,bytes,0.000000,266.000000,1.500090e+06,0,0.406188,ok
Total Length of Bwd Packets,float64,bytes,0.000000,0.000000,1.500090e+06,0,0.627529,ok
Fwd Packet Length Max,float64,bytes,0.000000,261.000000,1.460000e+03,0,0.406188,ok
Fwd Packet Length Min,float64,bytes,0.000000,0.000000,7.700000e+01,0,0.944466,ok
Fwd Packet Length Mean,float64,bytes,0.000000,56.000000,1.446567e+03,0,0.406188,ok
Fwd Packet Length Std,float64,bytes,0.000000,118.500000,4.558503e+02,0,0.461722,ok
Bwd Packet Length Max,float64,bytes,0.000000,0.000000,1.460000e+03,0,0.627529,ok


Feature compatibility report saved; flows: 2521


,Flow Duration,Total Fwd Packets,Total Backward Packets,Flow Packets/s,RST Flag Count,SYN Flag Count
true_label,,,,,,
DoS,48588.310091,4.0,3.0,137.584484,1.0,2.0
PortScan,8381.332115,1.0,1.0,238.625552,1.0,1.0
normal,336097.856617,10.0,9.0,202.379516,0.0,2.0


In [15]:
cfg.data_dir = '/content/cicids2017'
dp = data.DatasetProcessor(cfg)

# Re-discover files in the structured folder
files = dp.find_files()
print(len(files), 'CSV files')
display(dp.inspect_file(files[0]).iloc[:3, :12])

# Load dataset chronologically with clean class sampling
df = dp.load()
print(json.dumps({k: v for k, v in dp.report.items()}, indent=1, default=str))

# Split chronologically and verify distribution
sp = dp.split(df)
print({k: len(v) for k, v in sp.items()})
dist = pd.DataFrame({k: v['Label'].value_counts() for k, v in sp.items()}).fillna(0).astype(int)
display(dist)

px.bar(dist.reset_index().melt('Label'), x='Label', y='value', color='variable', barmode='group', log_y=True, title='Class distribution per split (log scale)').show()

q = df[FeatureSchema.names].quantile([.5, .99, .999, 1.0]).T
q['max/p99.9'] = q[1.0] / (q[.999] + 1e-9)
print('Outlier analysis (heaviest tails):')
display(q.sort_values('max/p99.9', ascending=False).head(10))

json.dump({k: (v if not isinstance(v, (np.integer,)) else int(v)) for k, v in dp.report.items()}, open(f'{cfg.work_dir}/dataset_report.json', 'w'), default=str)
tests.run_tests(only=['T11']);

8 CSV files


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443,192.168.10.5,54865,6,7/7/2017 3:30,3,2,0,12,0
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80,192.168.10.5,55054,6,7/7/2017 3:30,109,1,1,6,6
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80,192.168.10.5,55055,6,7/7/2017 3:30,52,1,1,6,6


08:19:06 INFO ids.data: loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
INFO:ids.data:loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
08:19:11 INFO ids.data: loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
INFO:ids.data:loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
08:19:23 INFO ids.data: loading Friday-WorkingHours-Morning.pcap_ISCX.csv
INFO:ids.data:loading Friday-WorkingHours-Morning.pcap_ISCX.csv
08:19:29 INFO ids.data: loading Monday-WorkingHours.pcap_ISCX.csv
INFO:ids.data:loading Monday-WorkingHours.pcap_ISCX.csv
08:19:50 INFO ids.data: loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
INFO:ids.data:loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
08:19:57 INFO ids.data: loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
INFO:ids.data:loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
08:20:00 INFO ids.data: loading Tuesday-WorkingHours.pcap_ISCX.csv
INFO:ids.data:loading T

{
 "raw_rows": 3119345,
 "raw_label_counts": {
  "BENIGN": 2273097,
  "nan": 288602,
  "DoS": 252661,
  "PortScan": 158930,
  "DDoS": 128027,
  "FTP-Patator": 7938,
  "SSH-Patator": 5897,
  "WebAttack": 2180,
  "Bot": 1966,
  "Infiltration": 36,
  "Heartbleed": 11
 },
 "nan_or_inf_cells": 14718702,
 "negative_cells": 3253,
 "constant_features": [],
 "rows_dropped_invalid": 291493,
 "duplicates_removed": 830555,
 "excluded_rare_classes": {
  "Infiltration": 35,
  "Heartbleed": 7
 },
 "clean_rows": 535173,
 "clean_label_counts": {
  "BENIGN": 200000,
  "DoS": 192727,
  "DDoS": 127997,
  "FTP-Patator": 5929,
  "SSH-Patator": 3153,
  "WebAttack": 2091,
  "PortScan": 1925,
  "Bot": 1351
 }
}


08:20:54 INFO ids.data: leakage checks passed (disjoint rows, per-class chronological order)
INFO:ids.data:leakage checks passed (disjoint rows, per-class chronological order)


{'train': 315753, 'val': 101686, 'test': 107038}


,train,val,test
Label,,,
BENIGN,118000,38000,40000
DoS,113709,36618,38546
DDoS,75519,24320,25600
FTP-Patator,3498,1127,1186
SSH-Patator,1860,600,631
WebAttack,1234,398,419
PortScan,1136,366,385
Bot,797,257,271


Outlier analysis (heaviest tails):


,0.5,0.99,0.999,1.0,max/p99.9
RST Flag Count,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+09
Flow Bytes/s,145.177628,2.111462e+06,8.650937e+06,1.980000e+09,2.288769e+02
Total Length of Bwd Packets,4511.000000,4.003136e+04,4.085731e+05,1.800000e+07,4.405577e+01
Bwd Packets/s,0.198048,1.290323e+04,6.666666e+04,2.000000e+06,3.000000e+01
Total Length of Fwd Packets,86.000000,6.137000e+03,2.658214e+04,7.234320e+05,2.721496e+01
Total Backward Packets,5.000000,4.100000e+01,2.490000e+02,6.681000e+03,2.683133e+01
Total Fwd Packets,5.000000,3.600000e+01,1.990000e+02,5.332000e+03,2.679397e+01
Active Min,0.000000,4.103897e+06,7.014638e+06,1.100000e+08,1.568149e+01
Active Mean,0.000000,4.480711e+06,7.015042e+06,1.100000e+08,1.568059e+01
Active Max,0.000000,6.414808e+06,9.419070e+06,1.100000e+08,1.167844e+01


test (tests.T11_DatasetPreprocessing.test) ... 08:20:56 INFO ids.sim: start normal#1 {'rate': 8}
INFO:ids.sim:start normal#1 {'rate': 8}
08:20:56 INFO ids.sim: start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 300), 'interval': 0.004}
INFO:ids.sim:start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 300), 'interval': 0.004}
ERROR

ERROR: test (tests.T11_DatasetPreprocessing.test)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/ids_src/tests.py", line 70, in test
    cfg = Config(data_dir=d, min_class_rows=10); dp = data.DatasetProcessor(cfg); df = dp.load()
                                                                                       ~~~~~~~^^
  File "/content/ids_src/data.py", line 42, in load
    for f in self.find_files():
             ~~~~~~~~~~~~~~~^^
  File "/content/ids_src/data.py", line 34, in find_files
    raise FileNotFoundError(f"No CIC-IDS2017 CSV files under {self.c

In [17]:
mt = models.ModelTrainer(cfg)
imb = mt.compare_imbalance(sp['train'], sp['val'], core.Preprocessor().fit(sp['train'][FeatureSchema.names]), sorted(sp['train']['Label'].unique(), key=lambda c: (c != 'BENIGN', c)))
print('Imbalance strategies (small RF, validation macro-F1):'); display(imb)
t0 = time.time(); H = mt.train(sp); print('training done in %.0fs' % (time.time() - t0), mt.timings)
print(json.dumps(H.cfg_dict, indent=1, default=str)); H.save(cfg.work_dir); print('saved to', cfg.work_dir)

Imbalance strategies (small RF, validation macro-F1):


,macro_f1
none,0.850819
class_weight,0.850759
undersample(cap 5k)+none,0.848570
oversample(to median),0.844420


08:29:29 INFO ids.models: hybrid calibrated: {'selected': 'blend', 'combiner_val_tpr': {'weighted': 0.9931855294560985, 'blend': 0.9941904283381485, 'maxrule': 0.9924004522044969}, 'w_rf': 0.6000000000000001, 'class_bias': [2.0, 0.5, 3.0, 3.0, 1.0, 1.0, 1.0, 5.0], 'tau_review': 0.809423354516898, 'tau_high': 0.8266036303518169, 's_thr': 0.84, 'signal_quantile': 0.99, 'combiner': {'kind': 'blend'}}
INFO:ids.models:hybrid calibrated: {'selected': 'blend', 'combiner_val_tpr': {'weighted': 0.9931855294560985, 'blend': 0.9941904283381485, 'maxrule': 0.9924004522044969}, 'w_rf': 0.6000000000000001, 'class_bias': [2.0, 0.5, 3.0, 3.0, 1.0, 1.0, 1.0, 5.0], 'tau_review': 0.809423354516898, 'tau_high': 0.8266036303518169, 's_thr': 0.84, 'signal_quantile': 0.99, 'combiner': {'kind': 'blend'}}


training done in 151s {'rf': 39.501811027526855, 'xgb': 54.73822522163391, 'iforest': 0.6218907833099365, 'autoencoder': 50.108086824417114}
{
 "selected": "blend",
 "combiner_val_tpr": {
  "weighted": 0.9931855294560985,
  "blend": 0.9941904283381485,
  "maxrule": 0.9924004522044969
 },
 "w_rf": 0.6000000000000001,
 "class_bias": [
  2.0,
  0.5,
  3.0,
  3.0,
  1.0,
  1.0,
  1.0,
  5.0
 ],
 "tau_review": 0.809423354516898,
 "tau_high": 0.8266036303518169,
 "s_thr": 0.84,
 "signal_quantile": 0.99,
 "combiner": {
  "kind": "blend"
 }
}
saved to /content/ids_artifacts


In [19]:
res = evalx.evaluate_all(H, sp['test'], H.val2)
comp = pd.DataFrame(res['comparison']).set_index('system'); display(comp[['accuracy','precision','recall','F1','FPR','FNR','PR_AUC','ROC_AUC','macro_F1','weighted_F1','HIGH_only_recall','HIGH_only_FPR','unknown_flagged']].round(4))
display(pd.DataFrame(res['per_class_recall']).round(4));
for k, v in res['confusion'].items():
    px.imshow(np.array(v['matrix']), x=v['labels'], y=v['labels'], text_auto=True, color_continuous_scale='Blues', title=f'Confusion matrix – {k}', labels=dict(x='Predicted', y='True')).show()
fig = go.Figure(); fig2 = go.Figure()
for n, cv in res['curves'].items(): fig.add_scatter(x=cv['recall'], y=cv['precision'], name=n); fig2.add_scatter(x=cv['fpr'], y=cv['tpr'], name=n)
fig.update_layout(title='Precision-Recall (attack vs benign)', xaxis_title='Recall', yaxis_title='Precision').show(); fig2.update_layout(title='ROC', xaxis_title='FPR', yaxis_title='TPR').show()

,accuracy,precision,recall,F1,FPR,FNR,PR_AUC,ROC_AUC,macro_F1,weighted_F1,HIGH_only_recall,HIGH_only_FPR,unknown_flagged
system,,,,,,,,,,,,,
Random Forest,0.9104,0.9961,0.8603,0.9232,0.0056,0.1397,0.9946,0.9909,0.7151,0.8977,NaN,NaN,NaN
XGBoost,0.9107,0.9960,0.8608,0.9235,0.0058,0.1392,0.9978,0.9964,0.6989,0.8948,NaN,NaN,NaN
Supervised ensemble,0.9171,0.9962,0.8709,0.9294,0.0056,0.1291,0.9948,0.9943,0.7161,0.9056,NaN,NaN,NaN
Isolation Forest,0.3872,0.5901,0.0705,0.1260,0.0821,0.9295,0.7870,0.8329,NaN,NaN,NaN,NaN,NaN
Autoencoder,0.6766,0.9014,0.5430,0.6777,0.0996,0.4570,0.8368,0.8834,NaN,NaN,NaN,NaN,NaN
Hybrid IDS (REVIEW+HIGH = alert),0.8655,0.9523,0.8266,0.8850,0.0694,0.1734,0.9907,0.9840,0.7101,0.8649,0.3866,0.0004,2764.0


,Random Forest,XGBoost,Supervised ensemble,Hybrid IDS (REVIEW+HIGH = alert),Hybrid (any alert)
BENIGN,0.9944,0.9942,0.9944,0.9306,NaN
Bot,0.0000,0.0000,0.0000,0.0000,0.0000
DDoS,0.9971,0.9973,0.9975,0.9969,0.9971
DoS,0.7372,0.7277,0.7603,0.6853,0.7112
FTP-Patator,0.9941,0.9941,0.9941,0.9924,0.9983
PortScan,0.8234,0.8182,0.8182,0.8130,0.8364
SSH-Patator,0.7734,0.7686,0.7623,0.7591,0.9097
WebAttack,0.8998,0.9236,0.9284,0.9260,0.9451


In [20]:
abl = evalx.ablation(H, H.val2, sp['test']); print('Ablation (weights tuned on val2; test measured once):'); display(pd.DataFrame(abl).round(4))
lat = evalx.latency_throughput(H, sp['test']); print('Latency/throughput:', lat)
print('Combiner candidates (validation TPR @ benign FPR=%.3f):' % cfg.target_fpr_review, H.combiner_scores, '-> selected:', H.cfg_dict['selected'])

Ablation (weights tuned on val2; test measured once):


,components,weights,val_TPR,test_recall,test_FPR,PR_AUC,ROC_AUC
0,S,[1.0],0.9930,0.7749,0.0026,0.9948,0.9943
1,I,[1.0],0.0845,0.0696,0.0819,0.7875,0.8329
2,A,[1.0],0.8338,0.5395,0.0995,0.8394,0.8834
3,S+I,"[0.6, 0.4]",0.9932,0.7777,0.0033,0.9944,0.9910
4,S+A,"[0.7, 0.3]",0.9932,0.7830,0.0030,0.9956,0.9927
5,I+A,"[0.0, 1.0]",0.8338,0.5395,0.0995,0.8394,0.8834
6,S+I+A,"[0.6, 0.4, 0.0]",0.9932,0.7777,0.0033,0.9944,0.9910


Latency/throughput: {'throughput_flows_per_s': 24714.638502427388, 'single_flow_latency_ms': 99.82597827911377}
Combiner candidates (validation TPR @ benign FPR=0.020): {'weighted': 0.9931855294560985, 'blend': 0.9941904283381485, 'maxrule': 0.9924004522044969} -> selected: blend


In [21]:
sm = evalx.stress_feature_mixing(H, sp['test'].sample(min(60000, len(sp['test'])), random_state=0)); print('Feature-mixing stress:'); display(pd.DataFrame(sm).round(4))
lo = evalx.leave_one_class_out(H, sp['train'], sp['test'], cfg); print('Leave-one-class-out:'); display(pd.DataFrame(lo).round(3))

Feature-mixing stress:


,mix_lambda,hybrid_recall,supervised_only_recall,anomaly_only_recall,hybrid_FNR
0,0.00,0.8261,0.8497,0.3970,0.1739
1,0.25,0.0415,0.0741,0.3731,0.9585
2,0.50,0.0060,0.0062,0.3665,0.9940
3,0.75,0.0227,0.0005,0.0847,0.9773


Leave-one-class-out:


,held_out_class,n,supervised_detects,both_anomaly_signals,either_anomaly_signal,caught_by_hybrid_logic
0,Bot,271,0.000,0.000,0.000,0.000
1,DDoS,25600,0.572,0.010,0.413,0.572
2,DoS,38546,0.030,0.012,0.403,0.041
3,FTP-Patator,1186,0.006,0.000,0.002,0.006
4,PortScan,385,0.164,0.013,0.574,0.177
5,SSH-Patator,631,0.170,0.000,0.008,0.170
6,WebAttack,419,0.076,0.000,0.000,0.076


In [22]:
def sim_trial(kind, seed=3, dur=40, **p):
    b = backend.IDSBackend(H, cfg, seed); b.ctl.start('normal', dur + 10, rate=4); b.ctl.start(kind, dur, **p); b.run(dur + 15); f = b.flows
    lab = {'portscan':'PortScan','dos':'DoS','ddos':'DDoS','ssh_brute':'SSH-Patator','ftp_brute':'FTP-Patator','web_attack':'WebAttack','botnet':'Bot'}[kind]
    a, n = f[f.true_label == lab], f[f.true_label == 'normal']; first = a[a.final_status != 'NORMAL'].start_ts.min() if len(a) else np.nan
    return dict(scenario=kind, params=str(p), attack_flows=len(a), recall=float((a.final_status != 'NORMAL').mean()) if len(a) else np.nan, high_share=float((a.final_status == 'HIGH').mean()) if len(a) else np.nan,
                benign_flows=len(n), benign_FPR=float((n.final_status != 'NORMAL').mean()) if len(n) else np.nan, detect_delay_s=float(first - a.start_ts.min()) if len(a) and first == first else np.nan)
rows = []
for kind in ['portscan', 'dos', 'ssh_brute', 'ftp_brute', 'botnet']:
    for inten in [1.0, 0.5, 0.25, 0.1]: rows.append(sim_trial(kind, intensity=inten))
for kind in ['dos', 'ssh_brute']: rows.append(sim_trial(kind, ramp=30.0))                 # gradual onset
for srcs in (['a1'], ['a1', 'a2'], ['a1', 'a2', 'a3']): rows.append(sim_trial('ddos', src=srcs, rate=120.0))   # multi-source
sim_stress = pd.DataFrame(rows); display(sim_stress.round(3))

08:37:19 INFO ids.sim: start normal#1 {'rate': 4}
INFO:ids.sim:start normal#1 {'rate': 4}
08:37:19 INFO ids.sim: start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004, 'intensity': 1.0}
INFO:ids.sim:start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004, 'intensity': 1.0}
08:37:36 INFO ids.sim: start normal#1 {'rate': 4}
INFO:ids.sim:start normal#1 {'rate': 4}
08:37:36 INFO ids.sim: start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004, 'intensity': 0.5}
INFO:ids.sim:start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004, 'intensity': 0.5}
08:37:42 INFO ids.sim: start normal#1 {'rate': 4}
INFO:ids.sim:start normal#1 {'rate': 4}
08:37:42 INFO ids.sim: start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004, 'intensity': 0.25}
INFO:ids.sim:start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.00

,scenario,params,attack_flows,recall,high_share,benign_flows,benign_FPR,detect_delay_s
0,portscan,{'intensity': 1.0},1024,0.000,0.0,193,0.093,NaN
1,portscan,{'intensity': 0.5},1024,0.000,0.0,191,0.099,NaN
2,portscan,{'intensity': 0.25},1024,0.000,0.0,189,0.095,NaN
3,portscan,{'intensity': 0.1},991,0.000,0.0,190,0.132,NaN
4,dos,{'intensity': 1.0},4721,0.249,0.0,190,0.095,0.065
5,dos,{'intensity': 0.5},2330,0.245,0.0,161,0.093,0.000
6,dos,{'intensity': 0.25},1249,0.266,0.0,183,0.104,0.000
7,dos,{'intensity': 0.1},512,0.238,0.0,172,0.099,0.277
8,ssh_brute,{'intensity': 1.0},83,0.000,0.0,167,0.126,NaN
9,ssh_brute,{'intensity': 0.5},38,0.000,0.0,205,0.112,NaN


In [23]:
# Domain gap: simulator flows vs CIC test flows (mean KS over log1p features; 0 = identical distributions)
sf = []
for i, (k, p) in enumerate([('normal', {}), ('portscan', {}), ('dos', dict(rate=120)), ('ddos', {}), ('ftp_brute', {}), ('ssh_brute', {}), ('web_attack', {}), ('botnet', dict(period=3.0))]):
    s_, c_ = tests.mk(100 + i); c_.start('normal', 60, rate=5)
    if k != 'normal': c_.start(k, 40, **p)
    c_.run(70); f_ = tests.flows_of(s_); sf.append(f_)
SF = pd.concat(sf); gap = evalx.ks_domain_gap(sp['test'], SF, {'normal':'BENIGN','PortScan':'PortScan','DoS':'DoS','DDoS':'DDoS','FTP-Patator':'FTP-Patator','SSH-Patator':'SSH-Patator','WebAttack':'WebAttack','Bot':'Bot'}); display(gap)

08:44:09 INFO ids.sim: start normal#1 {'rate': 5}
INFO:ids.sim:start normal#1 {'rate': 5}
08:44:18 INFO ids.sim: start normal#1 {'rate': 5}
INFO:ids.sim:start normal#1 {'rate': 5}
08:44:18 INFO ids.sim: start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004}
INFO:ids.sim:start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004}
08:44:22 INFO ids.sim: start normal#1 {'rate': 5}
INFO:ids.sim:start normal#1 {'rate': 5}
08:44:22 INFO ids.sim: start dos#2 {'src': 'a1', 'dst': 'web', 'rate': 120}
INFO:ids.sim:start dos#2 {'src': 'a1', 'dst': 'web', 'rate': 120}
08:44:28 INFO ids.sim: start normal#1 {'rate': 5}
INFO:ids.sim:start normal#1 {'rate': 5}
08:44:28 INFO ids.sim: start ddos#2 {'src': ['a1', 'a2', 'a3'], 'dst': 'web', 'rate': 200.0}
INFO:ids.sim:start ddos#2 {'src': ['a1', 'a2', 'a3'], 'dst': 'web', 'rate': 200.0}
08:44:39 INFO ids.sim: start normal#1 {'rate': 5}
INFO:ids.sim:start normal#1 {'rate': 5}
08:44:39 INFO ids.

,sim_class,cic_class,n_sim,n_cic,mean_KS,worst_features
0,normal,BENIGN,2403,40000,0.387882,"[Active Max, Active Min, Flow IAT Min]"
1,PortScan,PortScan,1024,385,0.595040,"[Flow IAT Min, SYN Flag Count, RST Flag Count]"
2,DoS,DoS,4633,38546,0.597623,"[Fwd PSH Flags, SYN Flag Count, ACK Flag Count]"
3,DDoS,DDoS,8008,25600,0.613202,"[Fwd PSH Flags, SYN Flag Count, ACK Flag Count]"
4,FTP-Patator,FTP-Patator,117,1186,0.635888,"[Active Mean, Active Max, Active Min]"
5,SSH-Patator,SSH-Patator,70,631,0.695952,"[Active Mean, Active Max, Active Min]"
6,WebAttack,WebAttack,318,419,0.774430,"[Active Mean, Active Max, Active Min]"
7,Bot,Bot,14,271,0.829319,"[Active Mean, Active Max, Active Min]"


In [24]:
# Save every measured result for the dashboard (page 7)
out = dict(res); out.update(ablation=abl, stress_feature_mixing=sm, leave_one_class_out=lo, sim_stress=json.loads(sim_stress.to_json(orient='records')), latency=lat,
           hybrid_config=json.loads(json.dumps(H.cfg_dict, default=str)), domain_gap=json.loads(gap.to_json(orient='records')), imbalance=json.loads(imb.reset_index().to_json(orient='records')), train_timings=mt.timings)
json.dump(out, open(f'{cfg.work_dir}/metrics.json', 'w'), default=lambda o: o.item() if hasattr(o, 'item') else str(o)); print('metrics.json written')

metrics.json written


In [25]:
B = backend.IDSBackend(H, cfg, seed=cfg.seed)
def show(n=8):
    t = B.alerts.table();
    if len(t): display(t[['id','severity','src_ip','dst_ip','category','risk','count','agreement','action']].tail(n))
def alert_card(a):
    m = a['models']; print(f"SOURCE: {a['src_ip']}\nDESTINATION: {a['dst_ip']}\nSTATUS: {a['severity']} ({a['status_level']})\nPREDICTION: {a['category']}\nRISK: {a['risk']:.3f}")
    print(f"RANDOM FOREST: {m['rf_class']} (attack p={m['rf_attack_p']:.3f})\nXGBOOST: {m['xgb_class']} (attack p={m['xgb_attack_p']:.3f})\nISOLATION FOREST: benign-CDF quantile {m['I_iforest']:.3f}\nAUTOENCODER: recon error {m['ae_recon_error']:.4f} (quantile {m['A_ae']:.3f})")
    print('MODEL AGREEMENT:', a['agreement']); print('EVIDENCE:'); [print(' -', e) for e in a['evidence']]; print('RECOMMENDED:', a['action'])
print('=== SCENARIO 1: NORMAL NETWORK ==='); B.ctl.start('normal', 60, rate=5); B.run(65)
print(B.summary()); display(B.flows.groupby(['proto','final_status']).size().unstack(fill_value=0)); print('benign flows flagged:', B.detection_summary('normal'))

08:50:07 INFO ids.sim: start normal#1 {'rate': 5}
INFO:ids.sim:start normal#1 {'rate': 5}


=== SCENARIO 1: NORMAL NETWORK ===
{'packets': 26238, 'flows': 297, 'normal': 267, 'suspicious': 30, 'alerts': 7, 'clock': 66.0, 'active_scenarios': []}


final_status,NORMAL,REVIEW
proto,,
ICMP,17,0
TCP,175,30
UDP,75,0


benign flows flagged: {'flows': 297, 'alerted': 0.10101010101010101, 'high': 0.0, 'categories': {'BENIGN': 267, 'UNKNOWN_ANOMALY': 30}}


In [26]:
print('=== SCENARIO 2: PORT SCAN ==='); B.ctl.start('portscan', 30, src='a1', dst='web'); B.run(35)
print(B.detection_summary('PortScan')); a = B.alerts.table(); ps = a[a.category.eq('PortScan')]
if len(ps): alert_card(B.alerts.alerts[int(ps.iloc[0]['id'])])
else: print('No PortScan-labelled alert – inspect categories:', B.flows[B.flows.true_label=='PortScan'].category.value_counts().to_dict())

08:50:42 INFO ids.sim: start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004}
INFO:ids.sim:start portscan#2 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004}


=== SCENARIO 2: PORT SCAN ===
{'flows': 1024, 'alerted': 0.0, 'high': 0.0, 'categories': {'BENIGN': 1024}}
No PortScan-labelled alert – inspect categories: {'BENIGN': 1024}


In [36]:
print('=== SCENARIO 3: DoS / DDoS-like ==='); t0 = B.sim.clock
B.ctl.start('dos', 15, src='a2', dst='web', rate=120); B.run(20); B.ctl.start('ddos', 15, rate=120); B.run(20)
f = B.flows[B.flows.start_ts >= t0]; rate = f.groupby((f.start_ts // 5) * 5).size().rename('flows per 5 s'); display(rate.to_frame().T)
print('DoS:', B.detection_summary('DoS')); print('DDoS:', B.detection_summary('DDoS'))
display(f[f.true_label.isin(['DoS','DDoS'])][['rf_attack_p','xgb_attack_p','I_iforest','A_ae','S_sup','risk']].describe().loc[['mean','50%','max']].round(3))

08:57:41 INFO ids.sim: start dos#12 {'src': 'a2', 'dst': 'web', 'rate': 120}
INFO:ids.sim:start dos#12 {'src': 'a2', 'dst': 'web', 'rate': 120}


=== SCENARIO 3: DoS / DDoS-like ===


08:57:51 INFO ids.sim: start ddos#13 {'src': ['a1', 'a2', 'a3'], 'dst': 'web', 'rate': 120}
INFO:ids.sim:start ddos#13 {'src': ['a1', 'a2', 'a3'], 'dst': 'web', 'rate': 120}


start_ts,305.0,310.0,315.0,320.0,325.0,330.0,335.0,340.0
flows per 5 s,229,626,627,367,234,617,603,355


DoS: {'flows': 3635, 'alerted': 0.23961485557083906, 'high': 0.0, 'categories': {'BENIGN': 2764, 'UNKNOWN_ANOMALY': 871}}
DDoS: {'flows': 3610, 'alerted': 0.3398891966759003, 'high': 0.0, 'categories': {'BENIGN': 2383, 'UNKNOWN_ANOMALY': 1227}}


,rf_attack_p,xgb_attack_p,I_iforest,A_ae,S_sup,risk
mean,0.186,0.015,0.981,1.0,0.108,0.000
50%,0.162,0.006,0.988,1.0,0.142,0.000
max,0.543,0.110,0.996,1.0,0.344,0.004


In [35]:
print('=== SCENARIO 4: FALSE-POSITIVE TEST (legitimate but unusual traffic) ==='); n0 = len(B.flows)
B.ctl.start('burst', 25, src='c5', dst='web', rate=25); B.run(30); new = B.flows.iloc[n0:]; bl = new[new.true_label == 'normal']
print('model-level status:', bl.status.value_counts().to_dict(), '| after temporal/FP layer:', bl.final_status.value_counts().to_dict())
al = B.alerts.table(); mine = al[al.src_ip.eq('192.168.10.15')] if len(al) else al; display(mine[['id','severity','category','risk','count','investigation']] if len(mine) else 'No alert raised for the burst host')
if len(mine):
    aid = int(mine.iloc[0]['id']); B.alerts.set_status(aid, 'investigating'); B.fp.record(aid, 'false_positive', 'known backup/flash-crowd client', mine.iloc[0]['category']); B.fp.add_whitelist('192.168.10.15', 'verified legitimate burst')
    B.ctl.start('burst', 15, src='c5', rate=25); B.run(20); print('after whitelist, newest statuses:', B.flows.tail(100).final_status.value_counts().to_dict()); display(B.fp.summary())

08:57:27 INFO ids.sim: start burst#10 {'src': 'c5', 'dst': 'web', 'rate': 25}
INFO:ids.sim:start burst#10 {'src': 'c5', 'dst': 'web', 'rate': 25}


=== SCENARIO 4: FALSE-POSITIVE TEST (legitimate but unusual traffic) ===
model-level status: {'NORMAL': 620, 'REVIEW': 8} | after temporal/FP layer: {'NORMAL': 620, 'REVIEW': 8}


,id,severity,category,risk,count,investigation
1,2,WARNING,UNKNOWN_ANOMALY,0.000262,3,investigating
10,11,WARNING,UNKNOWN_ANOMALY,0.000256,6,new
13,14,WARNING,UNKNOWN_ANOMALY,0.000260,8,new


08:57:35 INFO ids.sim: start burst#11 {'src': 'c5', 'dst': 'web', 'rate': 25}
INFO:ids.sim:start burst#11 {'src': 'c5', 'dst': 'web', 'rate': 25}


after whitelist, newest statuses: {'NORMAL': 100}


,alert_id,verdict,note,category
0,2,false_positive,known backup/flash-crowd client,UNKNOWN_ANOMALY
1,2,false_positive,known backup/flash-crowd client,UNKNOWN_ANOMALY


In [29]:
print('=== SCENARIO 5: UNKNOWN ANOMALY (protocol behaviour absent from known classes) ===')
n0 = len(B.flows); B.ctl.start('unusual', 40, src='c4', dst='cnc', variant='icmp_tunnel'); B.ctl.start('unusual', 20, src='a3', dst='web', variant='xmas', rate=30); B.run(45)
u = B.flows.iloc[n0:]; u = u[u.true_label == 'Unknown']; print(len(u), 'flows of the unusual traffic'); display(u.groupby(['category','final_status']).size().rename('flows').to_frame())
display(u[['proto','I_iforest','A_ae','S_sup','risk','agreement']].head(5))

08:51:38 INFO ids.sim: start unusual#7 {'src': 'c4', 'dst': 'cnc', 'variant': 'icmp_tunnel'}
INFO:ids.sim:start unusual#7 {'src': 'c4', 'dst': 'cnc', 'variant': 'icmp_tunnel'}
08:51:38 INFO ids.sim: start unusual#8 {'src': 'a3', 'dst': 'web', 'variant': 'xmas', 'rate': 30}
INFO:ids.sim:start unusual#8 {'src': 'a3', 'dst': 'web', 'variant': 'xmas', 'rate': 30}


=== SCENARIO 5: UNKNOWN ANOMALY (protocol behaviour absent from known classes) ===
593 flows of the unusual traffic


,,flows
category,final_status,
UNKNOWN_ANOMALY,REVIEW,593


,proto,I_iforest,A_ae,S_sup,risk,agreement
5903,ICMP,0.997842,0.999421,0.142045,0.000262,"RF=BENIGN, XGB=BENIGN, IF=anomalous, AE=elevat..."
5904,TCP,0.993211,0.999421,0.809524,0.760218,"RF=BENIGN, XGB=Bot, IF=anomalous, AE=elevated ..."
5905,TCP,0.992211,0.999421,0.617021,0.172648,"RF=BENIGN, XGB=Bot, IF=anomalous, AE=elevated ..."
5906,TCP,0.991000,0.999421,0.617021,0.171960,"RF=BENIGN, XGB=Bot, IF=anomalous, AE=elevated ..."
5907,TCP,0.991368,0.999421,0.809524,0.758877,"RF=BENIGN, XGB=Bot, IF=anomalous, AE=elevated ..."


In [30]:
print('=== SCENARIO 6: SECURITY RESPONSE ==='); atk = '10.0.0.101'; web = '192.168.20.10'
print('before:', B.resp.verify(atk, web, 'TCP', 80)); rid = B.resp.block_ip(atk, comment='confirmed port scan (admin approved)')
print('after ACL block:', B.resp.verify(atk, web, 'TCP', 80)); n0 = len(B.flows); B.ctl.start('portscan', 15, src='a1'); B.run(20)
print('new PortScan flows from blocked host:', int(((B.flows.iloc[n0:].src_ip == atk) & (B.flows.iloc[n0:].true_label == 'PortScan')).sum()), '| firewall denies logged:', len(B.sim.drop_log))
B.resp.unblock(rid); print('after unblock:', B.resp.verify(atk, web, 'TCP', 80))
B.resp.isolate('192.168.10.13'); print('isolated c3 ->', B.resp.verify('192.168.10.13', web, 'TCP', 80)); display(B.sim.topology_figure()); B.resp.release('192.168.10.13')
display(pd.DataFrame(B.resp.incidents))

08:52:00 INFO ids.sim: start portscan#9 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004}
INFO:ids.sim:start portscan#9 {'src': 'a1', 'dst': 'web', 'port_range': (1, 1024), 'interval': 0.004}


=== SCENARIO 6: SECURITY RESPONSE ===
before: {'reachable': True, 'reason': 'ok'}
after ACL block: {'reachable': False, 'reason': 'acl-deny'}
new PortScan flows from blocked host: 0 | firewall denies logged: 1025
after unblock: {'reachable': True, 'reason': 'ok'}
isolated c3 -> {'reachable': False, 'reason': 'isolated-or-down'}


,time,action,target,outcome,reason,rule_id,alert_id,auto
0,238.0,verify,10.0.0.101->192.168.20.10:80,STILL REACHABLE,ok,NaN,NaN,NaN
1,238.0,block_ip,10.0.0.101,applied,NaN,3.0,NaN,False
2,238.0,verify,10.0.0.101->192.168.20.10:80,BLOCKED,acl-deny,NaN,NaN,NaN
3,258.0,unblock,3,removed,NaN,NaN,NaN,NaN
4,258.0,verify,10.0.0.101->192.168.20.10:80,STILL REACHABLE,ok,NaN,NaN,NaN
5,258.0,isolate_host,192.168.10.13,applied,NaN,NaN,NaN,False
6,258.0,verify,192.168.10.13->192.168.20.10:80,BLOCKED,isolated-or-down,NaN,NaN,NaN
7,258.0,release_host,192.168.10.13,applied,NaN,NaN,NaN,NaN


In [31]:
print('=== SCENARIO 7: PERFORMANCE EVALUATION ==='); display(comp[['recall','FPR','FNR','F1','PR_AUC','macro_F1']].round(4))
print('Hybrid per-class recall (any alert):'); display(pd.Series(res['per_class_recall']['Hybrid (any alert)']).round(4).to_frame('recall'))
print('Ablation:'); display(pd.DataFrame(abl).round(4))
print('Read the tables, not the design intent: if the hybrid does not improve over the supervised ensemble here, that is the measured result.')

=== SCENARIO 7: PERFORMANCE EVALUATION ===


,recall,FPR,FNR,F1,PR_AUC,macro_F1
system,,,,,,
Random Forest,0.8603,0.0056,0.1397,0.9232,0.9946,0.7151
XGBoost,0.8608,0.0058,0.1392,0.9235,0.9978,0.6989
Supervised ensemble,0.8709,0.0056,0.1291,0.9294,0.9948,0.7161
Isolation Forest,0.0705,0.0821,0.9295,0.1260,0.7870,NaN
Autoencoder,0.5430,0.0996,0.4570,0.6777,0.8368,NaN
Hybrid IDS (REVIEW+HIGH = alert),0.8266,0.0694,0.1734,0.8850,0.9907,0.7101


Hybrid per-class recall (any alert):


,recall
Bot,0.0000
DDoS,0.9971
DoS,0.7112
FTP-Patator,0.9983
PortScan,0.8364
SSH-Patator,0.9097
WebAttack,0.9451


Ablation:


,components,weights,val_TPR,test_recall,test_FPR,PR_AUC,ROC_AUC
0,S,[1.0],0.9930,0.7749,0.0026,0.9948,0.9943
1,I,[1.0],0.0845,0.0696,0.0819,0.7875,0.8329
2,A,[1.0],0.8338,0.5395,0.0995,0.8394,0.8834
3,S+I,"[0.6, 0.4]",0.9932,0.7777,0.0033,0.9944,0.9910
4,S+A,"[0.7, 0.3]",0.9932,0.7830,0.0030,0.9956,0.9927
5,I+A,"[0.0, 1.0]",0.8338,0.5395,0.0995,0.8394,0.8834
6,S+I+A,"[0.6, 0.4, 0.0]",0.9932,0.7777,0.0033,0.9944,0.9910


Read the tables, not the design intent: if the hybrid does not improve over the supervised ensemble here, that is the measured result.


In [32]:
tests.CTX.update(h=H, cfg=cfg, split=sp)
r = tests.run_tests(); print('\nTOTAL:', r.testsRun, 'tests | failures:', len(r.failures), '| errors:', len(r.errors))

test (tests.T01_Topology.test) ... ok
test (tests.T02_HostComm.test) ... ok
test (tests.T03_RouterForwarding.test) ... ok
test (tests.T04_SwitchForwarding.test) ... ok
test (tests.T05_TCP.test) ... ok
test (tests.T06_UDP.test) ... ok
test (tests.T07_ICMP.test) ... ok
test (tests.T08_AttackGeneration.test) ... 08:52:52 INFO ids.sim: start portscan#1 {'src': 'a1', 'dst': 'web', 'port_range': (1, 200), 'interval': 0.004}
INFO:ids.sim:start portscan#1 {'src': 'a1', 'dst': 'web', 'port_range': (1, 200), 'interval': 0.004}
08:52:52 INFO ids.sim: start dos#1 {'src': 'a1', 'dst': 'web', 'rate': 100}
INFO:ids.sim:start dos#1 {'src': 'a1', 'dst': 'web', 'rate': 100}
08:52:55 INFO ids.sim: start ddos#1 {'src': ['a1', 'a2', 'a3'], 'dst': 'web', 'rate': 100}
INFO:ids.sim:start ddos#1 {'src': ['a1', 'a2', 'a3'], 'dst': 'web', 'rate': 100}
ok
test (tests.T09_FlowExtraction.test) ... ok
test (tests.T10_Schema.test) ... 08:52:57 INFO ids.sim: start normal#1 {'rate': 3.0}
INFO:ids.sim:start normal#1 {'r


TOTAL: 22 tests | failures: 0 | errors: 1


In [33]:
import shutil; os.makedirs('/content/drive/MyDrive/ids_artifacts_backup', exist_ok=True) if os.path.isdir('/content/drive/MyDrive') else None
if os.path.isdir('/content/drive/MyDrive'): shutil.copytree(cfg.work_dir, '/content/drive/MyDrive/ids_artifacts_backup', dirs_exist_ok=True); print('backed up to Drive')

backed up to Drive


In [34]:
import subprocess, secrets, re
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
PW = secrets.token_urlsafe(9); env = dict(os.environ, IDS_PASSWORD=PW, IDS_ARTIFACTS=cfg.work_dir)
st_proc = subprocess.Popen(['streamlit', 'run', '/content/ids_src/app.py', '--server.port', '8501', '--server.headless', 'true', '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'true'],
                           env=env, stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
tun = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True); url = None; t0 = time.time()
while time.time() - t0 < 60 and url is None:
    line = tun.stdout.readline(); m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line or ''); url = m.group(0) if m else None
print('Dashboard URL :', url); print('Admin password:', PW); print('Stop with: st_proc.terminate(); tun.terminate()   (tail /content/streamlit.log for errors)')

Dashboard URL : https://remedy-preparation-organization-blanket.trycloudflare.com
Admin password: lZhrOLEjNdrk
Stop with: st_proc.terminate(); tun.terminate()   (tail /content/streamlit.log for errors)
